In [1]:
import pandas as pd
import numpy as np

import os
import gurobipy as gp
from gurobipy import GRB

os.environ["GRB_LICENSE_FILE"] = "/opt/gurobi910/gurobi.lic"

In [2]:
path_folder = "Datasets/"
output_path = "Datasets/Outputs/"
path_claim_optClean = "ds_claim_optClean.xlsx"
path_adjuster_optClean = "ds_adjuster_optClean.xlsx"
path_productivityMatrix = "ds_productivityMatrix.xlsx"
path_timeMatrix = "ds_timeMatrix.xlsx"

In [3]:
ds_claim_optClean = pd.read_excel(
    path_folder+path_claim_optClean,
    sheet_name=0
)

ds_adjuster_optClean = pd.read_excel(
    path_folder+path_adjuster_optClean,
    sheet_name=0
)

ds_timeMatrix = pd.read_excel(
    path_folder+path_timeMatrix,
    sheet_name=0
)

ds_productivityMatrix = pd.read_excel(
    path_folder+path_productivityMatrix,
    sheet_name=0
)

In [4]:
# Step 1 : Ensure cleaning

svc = ds_productivityMatrix.rename(columns={
    "CAT Severity Code": "severity",
    "Skill Level": "skill_level"
}).copy()

svc["severity"] = pd.to_numeric(svc["severity"], errors="coerce")
svc["skill_level"] = pd.to_numeric(svc["skill_level"], errors="coerce")

min_skill_by_severity = (
    svc.dropna(subset=["severity", "skill_level"])
      .groupby("severity")["skill_level"]
      .min()
      .reset_index(name="min_skill")
      .sort_values("severity")
)

claims_calendar = ds_claim_optClean[[
    "Claim Number", "NOL Date", "Division", "CAT Severity Code",
    "SLA Days", "Due Date", "On-Site Handling", "Virtual Handling",
    "Lat", "Lon"
]].copy()

claims_calendar = claims_calendar.rename(columns={
    "Claim Number": "claim_id",
    "NOL Date": "arrival_date",
    "Division": "division",
    "CAT Severity Code": "severity",
    "SLA Days": "sla_days",
    "Due Date": "due_date",
    "On-Site Handling": "onsite",
    "Virtual Handling": "online",
    "Lat": "lat",
    "Lon": "lon",
})

# 2. Map Division
division_map = {"PI": "personal", "BI": "business"}
claims_calendar["claim_type"] = claims_calendar["division"].map(division_map)

unmapped = claims_calendar.loc[claims_calendar["claim_type"].isna(), "division"].dropna().unique()

# 3.Ensure datetimes
claims_calendar["arrival_date"] = pd.to_datetime(claims_calendar["arrival_date"], errors="coerce")
claims_calendar["due_date"] = pd.to_datetime(claims_calendar["due_date"], errors="coerce")

# 4. Create integer day indices
anchor = claims_calendar["arrival_date"].min()

claims_calendar["arrival_day"] = (claims_calendar["arrival_date"] - anchor).dt.days + 1
claims_calendar["due_day"] = (claims_calendar["due_date"] - anchor).dt.days + 1

# 5. Clean types / sanity checks
claims_calendar["severity"] = pd.to_numeric(claims_calendar["severity"], errors="coerce").astype("Int64")
claims_calendar["sla_days"] = pd.to_numeric(claims_calendar["sla_days"], errors="coerce").astype("Int64")

claims_calendar["onsite"] = claims_calendar["onsite"].fillna(False).astype(bool)
claims_calendar["online"] = claims_calendar["online"].fillna(False).astype(bool)

both_true = claims_calendar[claims_calendar["onsite"] & claims_calendar["online"]]

# 6. Keep only what we need going forward
claims_calendar = claims_calendar[[
    "claim_id", "claim_type", "severity",
    "arrival_date", "arrival_day",
    "due_date", "due_day",
    "sla_days", "onsite", "online",
    "lat", "lon"
]].copy()

# 7) Select + rename core columns
adjusters_core = ds_adjuster_optClean[[
    "TIES Id", "CL Skill Level", "PL Skill Level",
    "Will Travel", "City", "State", "Lat.Home", "Lon.Home"
]].copy()

adjusters_core = adjusters_core.rename(columns={
    "TIES Id": "adjuster_id",
    "CL Skill Level": "skill_business",
    "PL Skill Level": "skill_personal",
    "Will Travel": "will_travel",
    "City": "home_city",
    "State": "home_state",
    "Lat.Home": "home_lat",
    "Lon.Home": "home_lon",
})

# 8. Force numeric skills
adjusters_core["skill_business"] = pd.to_numeric(adjusters_core["skill_business"], errors="coerce").astype("Int64")
adjusters_core["skill_personal"] = pd.to_numeric(adjusters_core["skill_personal"], errors="coerce").astype("Int64")

# 9. Normalize will_travel to boolean robustly
if adjusters_core["will_travel"].dtype != bool:
    # Common cases: 0/1, "Yes"/"No", "TRUE"/"FALSE"
    w = adjusters_core["will_travel"].astype(str).str.strip().str.lower()
    adjusters_core["will_travel"] = w.isin(["true", "1", "yes", "y", "t"])
else:
    adjusters_core["will_travel"] = adjusters_core["will_travel"].fillna(False).astype(bool)

# 10. Add shift capacity in "days"
adjusters_core["shift_capacity_days"] = 1.0  # 8-hour shift = 1 workday in our Model B

# 11. Standardize IDs + types

claims_ids = claims_calendar.copy()
claims_ids["claim_id"] = claims_ids["claim_id"].astype(str)

adj_ids = adjusters_core.copy()
adj_ids["adjuster_id"] = adj_ids["adjuster_id"].astype(str)

# ensure numeric skills
adj_ids["skill_business"] = pd.to_numeric(adj_ids["skill_business"], errors="coerce")
adj_ids["skill_personal"] = pd.to_numeric(adj_ids["skill_personal"], errors="coerce")

# 12. Claim to Cluster membership

claim_cluster = ds_timeMatrix.rename(columns={
    "Claim Number": "claim_id",
    "Cluster": "cluster_id",
}).copy()

claim_cluster["claim_id"] = claim_cluster["claim_id"].astype(str)
claim_cluster["cluster_id"] = claim_cluster["cluster_id"].astype(str)

# 13. Keep travel metrics as-is (for later productivity loss)
claim_cluster["drive_seconds"] = pd.to_numeric(claim_cluster["drive_seconds"], errors="coerce")
claim_cluster["drive_miles"] = pd.to_numeric(claim_cluster.get("drive_miles", np.nan), errors="coerce")

# 14. Join claim attributes onto claim_cluster
claim_cluster = claim_cluster.merge(
    claims_ids[[
        "claim_id", "claim_type", "severity",
        "arrival_day", "due_day", "onsite", "online"
    ]],
    on="claim_id",
    how="left",
    validate="many_to_one"
)

# Online claims: travel is irrelevant
claim_cluster.loc[claim_cluster["online"].eq(True), ["drive_seconds", "drive_miles"]] = 0.0

# normalize claim severity numeric
claim_cluster["severity"] = pd.to_numeric(claim_cluster["severity"], errors="coerce")

# 15.Cluster summary (demand flags + counts)
cluster_summary = (
    claim_cluster.groupby("cluster_id", as_index=False)
    .agg(
        claims=("claim_id", "nunique"),
        onsite_claims=("onsite", "sum"),
        online_claims=("online", "sum"),
        earliest_arrival=("arrival_day", "min"),
        latest_due=("due_day", "max"),
    )
)

cluster_summary["onsite_claims"] = cluster_summary["onsite_claims"].astype(int)
cluster_summary["online_claims"] = cluster_summary["online_claims"].astype(int)
cluster_summary["has_onsite"] = cluster_summary["onsite_claims"] > 0

# 16. Productivity lookup table (service_days)

svc = ds_productivityMatrix.rename(columns={
    "CAT Severity Code": "severity",
    "Skill Level": "skill_level"
}).copy()
svc["severity"] = pd.to_numeric(svc["severity"], errors="coerce")
svc["skill_level"] = pd.to_numeric(svc["skill_level"], errors="coerce")
svc["service_days"] = pd.to_numeric(svc["service_days"], errors="coerce")

# 17. Within-cluster Claim -> Adjuster eligibility (Stage 2 feasibility)
#    Skill depends on claim_type:
#      - business claim  -> skill_business
#      - personal claim  -> skill_personal

within_cluster_claim_adjuster = claim_cluster[[
    "cluster_id", "claim_id", "claim_type", "severity",
    "arrival_day", "due_day", "onsite", "online",
    "drive_seconds", "drive_miles"
]].merge(
    adj_ids[["adjuster_id", "skill_personal", "skill_business", "will_travel"]],
    how="cross"
)

# skill determined by claim_type
within_cluster_claim_adjuster["skill_for_claim"] = np.where(
    within_cluster_claim_adjuster["claim_type"].eq("business"),
    within_cluster_claim_adjuster["skill_business"],
    within_cluster_claim_adjuster["skill_personal"]
)
within_cluster_claim_adjuster["skill_for_claim"] = pd.to_numeric(
    within_cluster_claim_adjuster["skill_for_claim"], errors="coerce"
)

# merge productivity using (severity, skill_for_claim)
within_cluster_claim_adjuster = within_cluster_claim_adjuster.merge(
    svc[["severity", "skill_level", "service_days"]],
    left_on=["severity", "skill_for_claim"],
    right_on=["severity", "skill_level"],
    how="left",
    validate="many_to_one"
)

# Eligibility at claim level (baseline)
within_cluster_claim_adjuster["eligible_claim"] = True
within_cluster_claim_adjuster.loc[within_cluster_claim_adjuster["service_days"].isna(), "eligible_claim"] = False
within_cluster_claim_adjuster.loc[within_cluster_claim_adjuster["skill_for_claim"].isna(), "eligible_claim"] = False
within_cluster_claim_adjuster.loc[within_cluster_claim_adjuster["skill_for_claim"] < 1, "eligible_claim"] = False

# Onsite claims require will_travel=True
within_cluster_claim_adjuster.loc[
    within_cluster_claim_adjuster["onsite"].eq(True) &
    within_cluster_claim_adjuster["will_travel"].eq(False),
    "eligible_claim"
] = False

# 18. Cluster -> Adjuster eligibility derived from claim feasibility (Stage 1 feasibility)
#    eligible_cluster=True iff adjuster can solve at least one claim in that cluster

cluster_adjuster_eligibility = (
    within_cluster_claim_adjuster
    .groupby(["cluster_id", "adjuster_id"], as_index=False)["eligible_claim"]
    .any()
    .rename(columns={"eligible_claim": "eligible_cluster"})
)

# enforce travel policy at cluster level too (belt + suspenders)
cluster_adjuster_eligibility = cluster_adjuster_eligibility.merge(
    cluster_summary[["cluster_id", "has_onsite"]],
    on="cluster_id",
    how="left"
).merge(
    adj_ids[["adjuster_id", "will_travel"]],
    on="adjuster_id",
    how="left"
)

cluster_adjuster_eligibility.loc[
    cluster_adjuster_eligibility["has_onsite"] &
    (~cluster_adjuster_eligibility["will_travel"]),
    "eligible_cluster"
] = False

cluster_adjuster_eligibility = cluster_adjuster_eligibility[[
    "cluster_id", "adjuster_id", "eligible_cluster"
]].copy()

# 19. Optional tightening: apply derived eligible_cluster back onto eligible_claim

within_cluster_claim_adjuster = within_cluster_claim_adjuster.merge(
    cluster_adjuster_eligibility,
    on=["cluster_id", "adjuster_id"],
    how="left",
    validate="many_to_one"
)

within_cluster_claim_adjuster.loc[
    within_cluster_claim_adjuster["eligible_cluster"].ne(True),
    "eligible_claim"
] = False

# keep final columns
within_cluster_claim_adjuster = within_cluster_claim_adjuster[[
    "cluster_id", "claim_id", "adjuster_id",
    "eligible_claim",
    "arrival_day", "due_day",
    "claim_type", "severity",
    "skill_for_claim",
    "service_days",
    "drive_seconds", "drive_miles"
]].copy()


# 20. Final QC: any claim with 0 eligible adjusters?

claim_feas = (
    within_cluster_claim_adjuster
    .groupby("claim_id")["eligible_claim"]
    .sum()
    .reset_index(name="n_eligible_adjusters")
)


H = 60

start_day = int(claims_calendar["arrival_day"].min())
end_day = start_day + H - 1

planning_days = pd.DataFrame({"day": range(start_day, end_day + 1)})

# Claims that arrive within the horizon (you can also include earlier backlog if you have it)
claims_in_horizon = claims_calendar.loc[
    (claims_calendar["arrival_day"] >= start_day) & (claims_calendar["arrival_day"] <= end_day),
    ["claim_id", "arrival_day", "due_day", "claim_type", "severity", "onsite", "online"]
].copy()

# --- make claim_id type consistent ---
claims_in_horizon["claim_id"] = claims_in_horizon["claim_id"].astype(str)

pairs2_elig = within_cluster_claim_adjuster.loc[
    within_cluster_claim_adjuster["eligible_claim"]
].copy()

# keep only claims in horizon
pairs2_elig = pairs2_elig.merge(
    claims_in_horizon[["claim_id"]],
    on="claim_id",
    how="inner",
    validate="many_to_many"
)

# Days in horizon
planning_days = pd.DataFrame({"day": range(start_day, end_day + 1)})

claim_day = claims_in_horizon.merge(planning_days, how="cross")

# cannot work before arrival
claim_day = claim_day[claim_day["day"] >= claim_day["arrival_day"]].copy()

# HARD SLA: cannot work after due_day
claim_day = claim_day[claim_day["day"] <= claim_day["due_day"]].copy()

pairs2_elig2 = pairs2_elig.drop(columns=["due_day"], errors="ignore")

acd2 = pairs2_elig2.merge(
    claim_day[["claim_id", "day", "due_day"]],
    on="claim_id",
    how="inner",
    validate="many_to_many"
)

acd2["late_days"] = (acd2["day"] - acd2["due_day"]).clip(lower=0)

acd2 = acd2.merge(
    claims_in_horizon[["claim_id", "onsite", "online"]],
    on="claim_id",
    how="left",
    validate="many_to_one"
)

def travel_multiplier(hours, threshold=2.5, step=0.5, loss=0.20, floor=0.0):
    """
    Additive productivity loss:
    - For every 'step' hours above threshold, productivity drops by 'loss'
    """
    hours = np.asarray(hours, dtype=float)
    excess = np.maximum(0.0, hours - threshold)
    k = np.floor(excess / step)
    mult = 1.0 - loss * k
    return np.maximum(floor, mult)


# -------------------------------------------------
# Travel productivity adjustment
# -------------------------------------------------

# Convert travel time to hours
acd2["travel_hours"] = acd2["drive_seconds"] / 3600.0

# Online claims have no travel
acd2.loc[acd2["online"], "travel_hours"] = 0.0

# Compute travel productivity multiplier
acd2["travel_mult"] = travel_multiplier(acd2["travel_hours"])

# Base productivity from service time
acd2["base_progress_per_day"] = 1.0 / acd2["service_days"]

# Final productivity used by the optimizer
acd2["progress_per_day"] = acd2["base_progress_per_day"] * acd2["travel_mult"]


# ============================================================
# PRE-SOLVE CLEANUP + GROUPED-RESOURCE PREP
# ============================================================

# --- 1) Upper bound on achievable progress per claim ---
max_progress_ub = (
    acd2.groupby("claim_id")["progress_per_day"]
        .sum()
        .reset_index(name="max_progress_upper_bound")
)

impossible = max_progress_ub.loc[
    max_progress_ub["max_progress_upper_bound"] < 1.0
].copy()

impossible["claim_id"] = impossible["claim_id"].astype(str)
impossible_ids = set(impossible["claim_id"])

print("Structurally impossible claims:", len(impossible_ids))

# --- 2) Remove structurally impossible claims from candidate rows ---
acd2["claim_id"] = acd2["claim_id"].astype(str)
acd2["adjuster_id"] = acd2["adjuster_id"].astype(str)
acd2["cluster_id"] = acd2["cluster_id"].astype(str)

acd2 = acd2.loc[
    ~acd2["claim_id"].isin(impossible_ids)
].copy()

print("acd2 rows after removing impossible claims:", len(acd2))
print("Remaining claims in acd2:", acd2["claim_id"].nunique())

# --- 3) Build claims_model only for claims that remain in the solve ---
claim_ids_kept = set(acd2["claim_id"].astype(str).unique())

claims_model = claims_in_horizon[[
    "claim_id", "severity", "arrival_day", "due_day", "online", "onsite"
]].copy()
claims_model["claim_id"] = claims_model["claim_id"].astype(str)

claims_model = claims_model.loc[
    claims_model["claim_id"].isin(claim_ids_kept)
].copy()

claim_cluster_map = claim_cluster[["claim_id", "cluster_id"]].drop_duplicates().copy()
claim_cluster_map["claim_id"] = claim_cluster_map["claim_id"].astype(str)
claim_cluster_map["cluster_id"] = claim_cluster_map["cluster_id"].astype(str)

claims_model = claims_model.merge(
    claim_cluster_map, on="claim_id", how="left", validate="one_to_one"
)

claims_model["severity"] = pd.to_numeric(claims_model["severity"], errors="coerce").fillna(1).astype(int)
claims_model["w"] = claims_model["severity"].astype(float)

print("Final acd2 rows:", len(acd2))
print("Final claims in acd2:", acd2["claim_id"].nunique())
print("Final claims_model rows:", len(claims_model))

# ============================================================
# BUILD GROUPED RESOURCES
# group = exact symmetry group:
#   (skill_business, skill_personal, will_travel, eligible_cluster_signature)
# ============================================================

# Cluster eligibility signature per adjuster
elig = cluster_adjuster_eligibility.loc[
    cluster_adjuster_eligibility["eligible_cluster"] == True
].copy()

elig["adjuster_id"] = elig["adjuster_id"].astype(str)
elig["cluster_id"] = elig["cluster_id"].astype(str)

cluster_signature = (
    elig.groupby("adjuster_id")["cluster_id"]
    .apply(lambda s: tuple(sorted(s.unique())))
    .reset_index(name="eligible_cluster_signature")
)

# Adjuster attributes
adj_group_base = adjusters_core[[
    "adjuster_id", "skill_business", "skill_personal", "will_travel"
]].copy()

adj_group_base["adjuster_id"] = adj_group_base["adjuster_id"].astype(str)
adj_group_base["skill_business"] = pd.to_numeric(adj_group_base["skill_business"], errors="coerce")
adj_group_base["skill_personal"] = pd.to_numeric(adj_group_base["skill_personal"], errors="coerce")
adj_group_base["will_travel"] = adj_group_base["will_travel"].astype(bool)

adj_group_base = adj_group_base.merge(cluster_signature, on="adjuster_id", how="left")
adj_group_base["eligible_cluster_signature"] = adj_group_base["eligible_cluster_signature"].apply(
    lambda x: x if isinstance(x, tuple) else tuple()
)

# Keep only adjusters that actually appear in acd2
active_adjusters = set(acd2["adjuster_id"].astype(str).unique())
adj_group_base = adj_group_base.loc[
    adj_group_base["adjuster_id"].isin(active_adjusters)
].copy()

# Exact group key
adj_group_base["group_key"] = list(zip(
    adj_group_base["skill_business"],
    adj_group_base["skill_personal"],
    adj_group_base["will_travel"],
    adj_group_base["eligible_cluster_signature"]
))

group_key_df = (
    adj_group_base[["group_key"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "group_num"})
)
group_key_df["group_id"] = group_key_df["group_num"].apply(lambda x: f"G{int(x)+1:03d}")

adj_group_base = adj_group_base.merge(
    group_key_df[["group_key", "group_id"]],
    on="group_key",
    how="left",
    validate="many_to_one"
)

# Group summary
group_summary = (
    adj_group_base.groupby("group_id", as_index=False)
    .agg(
        n_adjusters=("adjuster_id", "nunique"),
        skill_business=("skill_business", "first"),
        skill_personal=("skill_personal", "first"),
        will_travel=("will_travel", "first"),
        eligible_cluster_signature=("eligible_cluster_signature", "first")
    )
)

print("Number of grouped resources:", len(group_summary))
print("Total adjusters represented:", int(group_summary["n_adjusters"].sum()))
display(group_summary.head(20))

# Map adjusters -> groups into acd2
acd2_grouped = acd2.merge(
    adj_group_base[["adjuster_id", "group_id"]],
    on="adjuster_id",
    how="inner",
    validate="many_to_one"
).copy()

# Aggregate candidate work rows to (group, claim, day)
# Within each exact group, progress_per_day should be identical for a given claim/day
group_work = (
    acd2_grouped.groupby(
        ["group_id", "cluster_id", "claim_id", "day", "onsite", "online"],
        as_index=False
    )
    .agg(
        progress_per_day=("progress_per_day", "first"),
        n_original_rows=("adjuster_id", "nunique")
    )
)

print("Grouped candidate rows:", len(group_work))
print("Original acd2 rows:", len(acd2_grouped))

# Group-cluster eligibility
group_cluster_eligibility = (
    acd2_grouped.groupby(["group_id", "cluster_id"], as_index=False)
    .agg(eligible_cluster=("adjuster_id", "nunique"))
)
group_cluster_eligibility["eligible_cluster"] = True

# Group daily capacity table
capacity_group = (
    group_summary[["group_id", "n_adjusters"]]
    .assign(key=1)
    .merge(planning_days.assign(key=1), on="key")
    .drop(columns="key")
    .rename(columns={"n_adjusters": "capacity_units"})
)

print("capacity_group rows:", len(capacity_group))

Structurally impossible claims: 9
acd2 rows after removing impossible claims: 9023427
Remaining claims in acd2: 1053
Final acd2 rows: 9023427
Final claims in acd2: 1053
Final claims_model rows: 1053
Number of grouped resources: 19
Total adjusters represented: 602


,group_id,n_adjusters,skill_business,skill_personal,will_travel,eligible_cluster_signature
0,G001,20,0,1,True,"(0, 1, 10, 11, 12, 13, 14, 15, 17, 18, 2, 22, ..."
1,G002,63,0,3,True,"(0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
2,G003,101,0,2,True,"(0, 1, 10, 11, 12, 13, 14, 15, 17, 18, 2, 20, ..."
3,G004,74,3,3,True,"(0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
4,G005,107,1,3,True,"(0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
5,G006,55,1,1,True,"(0, 1, 10, 11, 12, 13, 14, 15, 17, 18, 2, 22, ..."
6,G007,18,1,2,True,"(0, 1, 10, 11, 12, 13, 14, 15, 17, 18, 2, 20, ..."
7,G008,33,4,4,True,"(0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
8,G009,42,2,3,True,"(0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
9,G010,17,2,2,True,"(0, 1, 10, 11, 12, 13, 14, 15, 17, 18, 2, 20, ..."


Grouped candidate rows: 290067
Original acd2 rows: 9023427
capacity_group rows: 1140


In [5]:
# ============================================================
# GUROBI MODEL (GROUPED RESOURCES)
# - x[g,c] = number of adjusters from group g deployed to cluster c
# - w[g,i,d] = number of adjusters from group g working claim i on day d
# - z[i] = claim completed
# - preserves cluster deployment logic and daily capacity
# - individual adjusters are assigned later within each group
# ============================================================

import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict

# ============================================================
# 1) Claim metadata
# ============================================================
claims_model_ids = claims_model.copy()

required_cols = ["claim_id", "severity", "arrival_day", "due_day", "onsite", "online", "cluster_id"]
missing = [c for c in required_cols if c not in claims_model_ids.columns]
if missing:
    raise KeyError(f"claims_model is missing required columns: {missing}")

claims_model_ids["claim_id"]    = claims_model_ids["claim_id"].astype(str)
claims_model_ids["cluster_id"]  = claims_model_ids["cluster_id"].astype(str)
claims_model_ids["online"]      = claims_model_ids["online"].astype(bool)
claims_model_ids["onsite"]      = claims_model_ids["onsite"].astype(bool)
claims_model_ids["due_day"]     = claims_model_ids["due_day"].astype(int)
claims_model_ids["arrival_day"] = claims_model_ids["arrival_day"].astype(int)

if "w" not in claims_model_ids.columns:
    claims_model_ids["w"] = pd.to_numeric(claims_model_ids["severity"], errors="coerce").fillna(1).astype(int)

claim_cluster_id = dict(zip(claims_model_ids["claim_id"], claims_model_ids["cluster_id"]))
claim_is_virtual = dict(zip(claims_model_ids["claim_id"], claims_model_ids["online"]))
claim_is_onsite  = dict(zip(claims_model_ids["claim_id"], claims_model_ids["onsite"]))
claim_due_day    = dict(zip(claims_model_ids["claim_id"], claims_model_ids["due_day"]))
claim_arr_day    = dict(zip(claims_model_ids["claim_id"], claims_model_ids["arrival_day"]))
claim_weight     = dict(zip(claims_model_ids["claim_id"], claims_model_ids["w"]))

I = sorted(claims_model_ids["claim_id"].unique().tolist())
D = sorted(planning_days["day"].astype(int).unique().tolist())

# ============================================================
# 2) Grouped candidate work table
# ============================================================
work_df = group_work[[
    "group_id", "cluster_id", "claim_id", "day", "progress_per_day", "onsite", "online"
]].copy()

work_df["group_id"] = work_df["group_id"].astype(str)
work_df["cluster_id"] = work_df["cluster_id"].astype(str)
work_df["claim_id"] = work_df["claim_id"].astype(str)
work_df["day"] = work_df["day"].astype(int)
work_df["progress_per_day"] = pd.to_numeric(work_df["progress_per_day"], errors="coerce").fillna(0.0).astype(float)

work_df = work_df.merge(
    claims_model_ids[["claim_id", "arrival_day", "due_day", "onsite", "online"]],
    on="claim_id",
    how="left",
    suffixes=("", "_claim"),
    validate="many_to_one"
)

work_df = work_df[
    (work_df["day"] >= work_df["arrival_day"]) &
    (work_df["day"] <= work_df["due_day"])
].copy()

work_df = work_df[work_df["progress_per_day"] > 0].copy()

work_df = work_df.drop(columns=["arrival_day", "due_day", "onsite_claim", "online_claim"], errors="ignore")

W_keys = list(zip(work_df["group_id"], work_df["claim_id"], work_df["day"]))
prog = {
    (g, i, d): float(p)
    for g, i, d, p in zip(
        work_df["group_id"],
        work_df["claim_id"],
        work_df["day"],
        work_df["progress_per_day"]
    )
}

G = sorted(group_summary["group_id"].astype(str).unique().tolist())

# ============================================================
# 3) Deployment keys x[g,c]
# ============================================================
clusters_with_onsite = set(cluster_summary.loc[cluster_summary["has_onsite"], "cluster_id"].astype(str))

x_elig = group_cluster_eligibility.copy()
x_elig["group_id"] = x_elig["group_id"].astype(str)
x_elig["cluster_id"] = x_elig["cluster_id"].astype(str)
x_elig = x_elig[x_elig["cluster_id"].isin(clusters_with_onsite)].copy()

X_keys = list(zip(x_elig["group_id"], x_elig["cluster_id"]))
X_set = set(X_keys)

# upper bound for x[g,c]
group_size = dict(zip(group_summary["group_id"].astype(str), group_summary["n_adjusters"].astype(int)))
x_ub = {(g, c): int(group_size[g]) for (g, c) in X_keys}

# ============================================================
# 4) Group daily capacity
# ============================================================
cap = {
    (g, d): int(c)
    for g, d, c in zip(
        capacity_group["group_id"].astype(str),
        capacity_group["day"].astype(int),
        capacity_group["capacity_units"].astype(int)
    )
}

# ============================================================
# 5) Group keys for constraints
# ============================================================
W_by_gd = defaultdict(list)         # (g,d) -> [(g,i,d)]
W_by_i  = defaultdict(list)         # i -> [(g,i,d)]
W_by_id = defaultdict(list)         # (i,d) -> [(g,i,d)]
W_by_gc_onsite = defaultdict(list)  # (g,c) -> [(g,i,d)] for onsite claims
W_by_gcd_onsite = defaultdict(list) # (g,c,d) -> [(g,i,d)] for onsite claims

for (g, i, d) in W_keys:
    W_by_gd[(g, d)].append((g, i, d))
    W_by_i[i].append((g, i, d))
    W_by_id[(i, d)].append((g, i, d))

    if claim_is_onsite.get(i, False):
        c = claim_cluster_id[i]
        W_by_gc_onsite[(g, c)].append((g, i, d))
        W_by_gcd_onsite[(g, c, d)].append((g, i, d))

# ============================================================
# 6) Build grouped model
# ============================================================
m = gp.Model("cluster_claim_scheduling_grouped")

m.Params.DisplayInterval = 10
m.Params.LogToConsole = 1
m.Params.TimeLimit = 1800
m.Params.MIPGap = 0.005

# ============================================================
# 7) Decision variables
# ============================================================
# integer number of group-g adjusters deployed to cluster c
x = m.addVars(
    X_keys,
    vtype=GRB.INTEGER,
    lb=0,
    ub=x_ub,
    name="x"
)

# integer number of group-g adjusters working claim i on day d
# at most MAX_TEAM people on a claim/day, so this is a natural upper bound
MAX_TEAM = 3
w_ub = {(g, i, d): int(min(group_size[g], MAX_TEAM)) for (g, i, d) in W_keys}

w = m.addVars(
    W_keys,
    vtype=GRB.INTEGER,
    lb=0,
    ub=w_ub,
    name="w"
)

# claim completion
z = m.addVars(I, vtype=GRB.BINARY, name="z")

# ============================================================
# 8) Constraints
# ============================================================

# (1) Each group can deploy at most its number of adjusters
X_by_g = defaultdict(list)
for (g, c) in X_keys:
    X_by_g[g].append((g, c))

for g in G:
    keys = X_by_g.get(g, [])
    if keys:
        m.addConstr(
            gp.quicksum(x[k] for k in keys) <= group_size[g],
            name=f"group_cluster_capacity_{g}"
        )

# (2) Daily capacity by group
for (g, d), keys in W_by_gd.items():
    m.addConstr(
        gp.quicksum(w[k] for k in keys) <= cap.get((g, d), 0),
        name=f"group_day_capacity_{g}_{d}"
    )

# (3) Onsite coupling by group-cluster-day
# total onsite workers from group g in cluster c on day d
# cannot exceed number of group-g adjusters deployed to c
for (g, c, d), keys in W_by_gcd_onsite.items():
    if (g, c) in X_set:
        m.addConstr(
            gp.quicksum(w[k] for k in keys) <= x[g, c],
            name=f"onsite_couple_{g}_{c}_{d}"
        )
    else:
        m.addConstr(
            gp.quicksum(w[k] for k in keys) == 0,
            name=f"onsite_forbid_{g}_{c}_{d}"
        )

# (4) Completion by due date
for i in I:
    due = claim_due_day[i]
    keys = [k for k in W_by_i.get(i, []) if k[2] <= due]
    if keys:
        m.addConstr(
            gp.quicksum(prog[k] * w[k] for k in keys) >= 1.0 * z[i],
            name=f"complete_{i}"
        )
    else:
        m.addConstr(z[i] == 0, name=f"no_work_possible_{i}")

# (4b) Cap progress with limited overshoot:
# allow at most one additional feasible worker-day chunk above 100%
TOL = 1e-6

for i in I:
    keys = W_by_i.get(i, [])
    if not keys:
        continue

    total_prog = gp.quicksum(prog[k] * w[k] for k in keys)

    # largest single worker-day contribution available for this claim
    max_chunk_i = max(prog[k] for k in keys)

    m.addConstr(
        total_prog <= 1.0 + max_chunk_i + TOL,
        name=f"cap_progress_{i}"
    )

# (5) No fake deployments:
# if x[g,c] people are deployed to c, then at least x[g,c] onsite workdays
# must happen in that cluster across the horizon
for (g, c) in X_keys:
    keys = W_by_gc_onsite.get((g, c), [])
    if keys:
        m.addConstr(
            gp.quicksum(w[k] for k in keys) >= x[g, c],
            name=f"no_fake_{g}_{c}"
        )
    else:
        m.addConstr(x[g, c] == 0, name=f"no_onsite_possible_{g}_{c}")

# (6) Max team per claim/day
for (i, d), keys in W_by_id.items():
    m.addConstr(
        gp.quicksum(w[k] for k in keys) <= MAX_TEAM,
        name=f"team_cap_{i}_{d}"
    )

# (NEW) Online work must also respect deployed capacity structure

for (g, d), keys in W_by_gd.items():
    deployed_capacity = gp.quicksum(x[g, c] for (gg, c) in X_keys if gg == g)

    m.addConstr(
        gp.quicksum(w[k] for k in keys) <= deployed_capacity,
        name=f"group_total_capacity_from_deploy_{g}_{d}"
    )

# ============================================================
# 9) Objective
# ============================================================
primary = gp.quicksum(claim_weight[i] * z[i] for i in I)
deploys = gp.quicksum(x[g, c] for (g, c) in X_keys)

M = 10_000
m.setObjective(M * primary - deploys, GRB.MAXIMIZE)

m.update()

# ============================================================
# 10) Optimize
# ============================================================
print("len(I) =", len(I))
print("len(G) =", len(G))
print("len(D) =", len(D))
print("len(X_keys) =", len(X_keys))
print("len(W_keys) =", len(W_keys))

print("Status before optimize:", m.Status)
m.optimize()
print("Status after optimize:", m.Status)

# ============================================================
# 11) Extract grouped results
# ============================================================
if m.SolCount > 0:
    print("ObjVal:", m.ObjVal)

    completed = [i for i in I if z[i].X > 0.5]
    print("Completed claims:", len(completed), "out of", len(I))

    deployed_pairs = [(g, c, int(round(x[g, c].X))) for (g, c) in X_keys if x[g, c].X > 0.5]
    print("Nonzero group deployments:", len(deployed_pairs))
    print("Total deployed adjusters:", sum(v for _, _, v in deployed_pairs))

    work_assignments = [(g, i, d, int(round(w[g, i, d].X))) for (g, i, d) in W_keys if w[g, i, d].X > 0.5]
    print("Nonzero grouped work assignments:", len(work_assignments))

    deploy_df = pd.DataFrame(deployed_pairs, columns=["group_id", "cluster_id", "n_deployed"])
    work_df_sol = pd.DataFrame(work_assignments, columns=["group_id", "claim_id", "day", "n_workers"])

    print("\nTop deployments:")
    print(deploy_df.sort_values("n_deployed", ascending=False).head(20))

    print("\nTop grouped work assignments:")
    print(work_df_sol.sort_values("n_workers", ascending=False).head(20))

else:
    print("No incumbent solution (SolCount=0).")

Set parameter TokenServer to value "hpcprdlic01.kennesaw.edu"
Set parameter DisplayInterval to value 10
Set parameter LogToConsole to value 1
Set parameter TimeLimit to value 1800
Set parameter MIPGap to value 0.005
len(I) = 1053
len(G) = 19
len(D) = 60
len(X_keys) = 528
len(W_keys) = 290067
Status before optimize: 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Red Hat Enterprise Linux 8.10 (Ootpa)")

CPU model: Intel(R) Xeon(R) Gold 6126 CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 22 physical cores, 22 logical processors, using up to 22 threads

Non-default parameters:
TimeLimit  1800
MIPGap  0.005
DisplayInterval  10

Optimize a model with 40659 rows, 291648 columns and 1882151 nonzeros (Max)
Model fingerprint: 0xec1c6d96
Model has 1581 linear objective coefficients
Variable types: 0 continuous, 291648 integer (1053 binary)
Coefficient statistics:
  Matrix range     [5e-02, 3e+00]
  Objective range  [1e+00, 5e+04]
  Bounds range     [1e+00, 1e+

In [6]:
# ============================================================
# MATERIALIZE REAL ADJUSTERS FROM GROUPED SOLUTION
# Assumes:
#   adj_group_base
#   group_summary
#   x, w, X_keys, W_keys
#   claims_model_ids
#   m
# ============================================================

import pandas as pd
from collections import defaultdict, deque

if m.SolCount == 0:
    raise RuntimeError("No incumbent solution available to materialize.")

# ------------------------------------------------------------
# 1) extract grouped nonzero solution
# ------------------------------------------------------------
deploy_df = pd.DataFrame(
    [(g, c, int(round(x[g, c].X))) for (g, c) in X_keys if x[g, c].X > 0.5],
    columns=["group_id", "cluster_id", "n_deployed"]
)
deploy_df["group_id"] = deploy_df["group_id"].astype(str)
deploy_df["cluster_id"] = deploy_df["cluster_id"].astype(str)

work_df_sol = pd.DataFrame(
    [(g, i, int(d), int(round(w[g, i, d].X))) for (g, i, d) in W_keys if w[g, i, d].X > 0.5],
    columns=["group_id", "claim_id", "day", "n_workers"]
)
work_df_sol["group_id"] = work_df_sol["group_id"].astype(str)
work_df_sol["claim_id"] = work_df_sol["claim_id"].astype(str)
work_df_sol["day"] = work_df_sol["day"].astype(int)

claims_meta = claims_model_ids.copy()
claims_meta["claim_id"] = claims_meta["claim_id"].astype(str)
claims_meta["cluster_id"] = claims_meta["cluster_id"].astype(str)

work_df_sol = work_df_sol.merge(
    claims_meta[["claim_id", "cluster_id", "onsite", "online", "severity", "arrival_day", "due_day"]],
    on="claim_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 2) build actual adjuster pools per group
# ------------------------------------------------------------
group_to_adjusters = (
    adj_group_base.groupby("group_id")["adjuster_id"]
    .apply(lambda s: sorted(map(str, s)))
    .to_dict()
)

# ------------------------------------------------------------
# 3) assign real adjusters to deployments
# one real adjuster -> one cluster max
# ------------------------------------------------------------
real_deployments = []
assigned_cluster_by_adjuster = {}

for _, row in deploy_df.sort_values(["group_id", "cluster_id"]).iterrows():
    g = row["group_id"]
    c = row["cluster_id"]
    need = int(row["n_deployed"])

    available = [a for a in group_to_adjusters.get(g, []) if a not in assigned_cluster_by_adjuster]

    if len(available) < need:
        raise RuntimeError(
            f"Not enough available adjusters in group {g} to assign {need} deployments to cluster {c}."
        )

    chosen = available[:need]
    for a in chosen:
        assigned_cluster_by_adjuster[a] = c
        real_deployments.append((a, g, c))

real_deployments_df = pd.DataFrame(real_deployments, columns=["adjuster_id", "group_id", "cluster_id"])

print("Real deployments created:", len(real_deployments_df))
display(real_deployments_df.head(20))

# ------------------------------------------------------------
# 4) assign actual adjusters to work rows (improved)
# rule:
#   - onsite claim must use adjusters deployed to that claim cluster
#   - online claim can use any deployed adjuster in same group
#   - one claim per adjuster per day
#   - assign day by day, onsite first, least-used workers first
# ------------------------------------------------------------
from collections import defaultdict

workers_by_group_cluster = defaultdict(list)
workers_by_group_any = defaultdict(list)

for _, row in real_deployments_df.iterrows():
    a = str(row["adjuster_id"])
    g = str(row["group_id"])
    c = str(row["cluster_id"])
    workers_by_group_cluster[(g, c)].append(a)
    workers_by_group_any[g].append(a)

for k in workers_by_group_cluster:
    workers_by_group_cluster[k] = sorted(workers_by_group_cluster[k])
for g in workers_by_group_any:
    workers_by_group_any[g] = sorted(set(workers_by_group_any[g]))

used_by_adjuster_day = defaultdict(set)
total_load_by_adjuster = defaultdict(int)
real_work_rows = []

assign_order = work_df_sol.copy()
assign_order["group_id"] = assign_order["group_id"].astype(str)
assign_order["claim_id"] = assign_order["claim_id"].astype(str)
assign_order["cluster_id"] = assign_order["cluster_id"].astype(str)
assign_order["day"] = assign_order["day"].astype(int)

# process one day at a time
for day, day_df in assign_order.groupby("day", sort=True):

    day_df = day_df.sort_values(
        ["onsite", "n_workers", "severity", "claim_id"],
        ascending=[False, False, False, True]
    ).copy()

    for _, row in day_df.iterrows():
        g = row["group_id"]
        i = row["claim_id"]
        d = int(row["day"])
        need = int(row["n_workers"])
        c = row["cluster_id"]
        onsite = bool(row["onsite"])
        online = bool(row["online"])

        if onsite:
            candidate_pool = workers_by_group_cluster.get((g, c), [])
        else:
            candidate_pool = workers_by_group_any.get(g, [])

        available = [a for a in candidate_pool if d not in used_by_adjuster_day[a]]

        # better balancing: use least-loaded workers first
        available = sorted(available, key=lambda a: (total_load_by_adjuster[a], a))

        if len(available) < need:
            raise RuntimeError(
                f"Could not materialize work row: group={g}, claim={i}, day={d}, "
                f"need={need}, available={len(available)}"
            )

        chosen = available[:need]

        for a in chosen:
            used_by_adjuster_day[a].add(d)
            total_load_by_adjuster[a] += 1
            real_work_rows.append((a, g, i, d, c, onsite, online))

real_work_assignments_df = pd.DataFrame(
    real_work_rows,
    columns=["adjuster_id", "group_id", "claim_id", "day", "cluster_id", "onsite", "online"]
)

print("Real work assignments created:", len(real_work_assignments_df))
display(real_work_assignments_df.head(20))

Real deployments created: 124


,adjuster_id,group_id,cluster_id
0,11226387,G002,10
1,11350447,G002,10
2,14343602,G002,17
3,16559402,G002,25
4,10312516,G003,11
5,10615629,G004,12
6,10925596,G004,12
7,11355772,G004,14
8,11494435,G004,22
9,14582357,G004,22


Real work assignments created: 2436


,adjuster_id,group_id,claim_id,day,cluster_id,onsite,online
0,59321509,G012,20407921,1,15,True,False
1,63175059,G012,20407921,1,15,True,False
2,63418508,G012,20407921,1,15,True,False
3,25351257,G008,43631890,1,15,True,False
4,27096178,G008,43631890,1,15,True,False
5,27328743,G008,43631890,1,15,True,False
6,11091566,G012,15394782,1,0,True,False
7,17566082,G012,15394782,1,0,True,False
8,11788873,G011,24098320,1,0,True,False
9,22143269,G011,24098320,1,0,True,False


## Tests

In [7]:
# ============================================================
# QA / SANITY CHECKS FOR GROUPED MODEL RESULTS
# Assumes you already solved the GROUPED model and have:
#   m, x, w, z, X_keys, W_keys, I, G
#   claims_model_ids
#   group_summary
#   capacity_group
#   group_work
#   group_cluster_eligibility
#   claim_cluster_id, claim_is_onsite, claim_is_virtual
#   claim_due_day, claim_arr_day
#   prog
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict

def _fmt_pass(p):
    return "PASS ✅" if p else "FAIL ❌"

def _qa_row(name, passed, metric=None, detail=None):
    return {"Test": name, "Status": _fmt_pass(passed), "Metric": metric, "Detail": detail}

qa_rows = []

# ------------------------------------------------------------
# 0) incumbent exists
# ------------------------------------------------------------
if m.SolCount == 0:
    qa_rows.append(_qa_row("Solution exists (SolCount > 0)", False, metric=f"SolCount={m.SolCount}",
                           detail="No incumbent solution."))
    qa_summary_grouped = pd.DataFrame(qa_rows)
    display(qa_summary_grouped)
    raise RuntimeError("No incumbent solution available for grouped QA.")
else:
    qa_rows.append(_qa_row("Solution exists (SolCount > 0)", True, metric=f"SolCount={m.SolCount}", detail=None))

# ------------------------------------------------------------
# 1) extract grouped solution
# ------------------------------------------------------------
claims_meta = claims_model_ids.copy()
claims_meta["claim_id"] = claims_meta["claim_id"].astype(str)
claims_meta["cluster_id"] = claims_meta["cluster_id"].astype(str)
claims_meta["onsite"] = claims_meta["onsite"].astype(bool)
claims_meta["online"] = claims_meta["online"].astype(bool)
claims_meta["arrival_day"] = claims_meta["arrival_day"].astype(int)
claims_meta["due_day"] = claims_meta["due_day"].astype(int)

group_size = dict(zip(group_summary["group_id"].astype(str), group_summary["n_adjusters"].astype(int)))
cap_group = {
    (g, d): int(c)
    for g, d, c in zip(
        capacity_group["group_id"].astype(str),
        capacity_group["day"].astype(int),
        capacity_group["capacity_units"].astype(int)
    )
}

x_sol = pd.DataFrame(
    [(g, c, float(x[g, c].X)) for (g, c) in X_keys],
    columns=["group_id", "cluster_id", "x_val"]
)
x_sol["group_id"] = x_sol["group_id"].astype(str)
x_sol["cluster_id"] = x_sol["cluster_id"].astype(str)
x_sol["x_int"] = x_sol["x_val"].round().astype(int)
x_sol_on = x_sol.loc[x_sol["x_int"] > 0].copy()

w_sol = pd.DataFrame(
    [(g, i, int(d), float(w[g, i, d].X)) for (g, i, d) in W_keys],
    columns=["group_id", "claim_id", "day", "w_val"]
)
w_sol["group_id"] = w_sol["group_id"].astype(str)
w_sol["claim_id"] = w_sol["claim_id"].astype(str)
w_sol["day"] = w_sol["day"].astype(int)
w_sol["w_int"] = w_sol["w_val"].round().astype(int)
w_sol_on = w_sol.loc[w_sol["w_int"] > 0].copy()

z_sol = pd.DataFrame(
    [(i, float(z[i].X)) for i in I],
    columns=["claim_id", "z_val"]
)
z_sol["claim_id"] = z_sol["claim_id"].astype(str)
z_sol["z_bin"] = (z_sol["z_val"] > 0.5).astype(int)

# merge claim metadata
w_sol_on = w_sol_on.merge(
    claims_meta[["claim_id", "cluster_id", "onsite", "online", "arrival_day", "due_day", "severity"]],
    on="claim_id",
    how="left",
    validate="many_to_one"
)

# progress must multiply by worker count
w_sol_on["progress_per_day"] = w_sol_on.apply(
    lambda r: float(prog.get((r["group_id"], r["claim_id"], int(r["day"])), 0.0)),
    axis=1
)
w_sol_on["progress"] = w_sol_on["progress_per_day"] * w_sol_on["w_int"]

# ------------------------------------------------------------
# 2) aggregates
# ------------------------------------------------------------
claim_progress = (
    w_sol_on.groupby("claim_id", as_index=False)["progress"]
    .sum()
    .rename(columns={"progress": "total_progress"})
)

claim_workers_total = (
    w_sol_on.groupby("claim_id", as_index=False)["w_int"]
    .sum()
    .rename(columns={"w_int": "total_worker_days"})
)

claim_days = (
    w_sol_on.groupby("claim_id", as_index=False)["day"]
    .nunique()
    .rename(columns={"day": "n_days_worked"})
)

team_by_claim_day = (
    w_sol_on.groupby(["claim_id", "day"], as_index=False)["w_int"]
    .sum()
    .rename(columns={"w_int": "team_size"})
)

claim_qa = (
    claims_meta[["claim_id", "cluster_id", "severity", "onsite", "online", "arrival_day", "due_day"]]
    .merge(z_sol[["claim_id", "z_bin"]], on="claim_id", how="left")
    .merge(claim_progress, on="claim_id", how="left")
    .merge(claim_workers_total, on="claim_id", how="left")
    .merge(claim_days, on="claim_id", how="left")
)

claim_qa["total_progress"] = claim_qa["total_progress"].fillna(0.0)
claim_qa["total_worker_days"] = claim_qa["total_worker_days"].fillna(0).astype(int)
claim_qa["n_days_worked"] = claim_qa["n_days_worked"].fillna(0).astype(int)

# ------------------------------------------------------------
# 3) grouped logical QA tests
# ------------------------------------------------------------

# T1: work only within [arrival_day, due_day]
viol_window = w_sol_on.loc[
    (w_sol_on["day"] < w_sol_on["arrival_day"]) | (w_sol_on["day"] > w_sol_on["due_day"]),
    ["group_id", "claim_id", "day", "arrival_day", "due_day"]
]
qa_rows.append(_qa_row(
    "Grouped work window respected",
    passed=(len(viol_window) == 0),
    metric=f"viol_rows={len(viol_window)}",
    detail=None if len(viol_window) == 0 else "See viol_window"
))

# T2: group deployment capacity
deploy_by_group = (
    x_sol_on.groupby("group_id", as_index=False)["x_int"]
    .sum()
    .rename(columns={"x_int": "n_deployed"})
)
deploy_by_group["group_capacity"] = deploy_by_group["group_id"].map(group_size)
viol_group_deploy = deploy_by_group.loc[deploy_by_group["n_deployed"] > deploy_by_group["group_capacity"]]
qa_rows.append(_qa_row(
    "Group deployment count <= group size",
    passed=(len(viol_group_deploy) == 0),
    metric=f"viol_groups={len(viol_group_deploy)}",
    detail=None if len(viol_group_deploy) == 0 else "See viol_group_deploy"
))

# T3: daily group capacity
work_by_gd = (
    w_sol_on.groupby(["group_id", "day"], as_index=False)["w_int"]
    .sum()
    .rename(columns={"w_int": "workers_assigned"})
)
work_by_gd["capacity_units"] = work_by_gd.apply(
    lambda r: int(cap_group.get((r["group_id"], int(r["day"])), 0)),
    axis=1
)
viol_group_day = work_by_gd.loc[work_by_gd["workers_assigned"] > work_by_gd["capacity_units"]]
qa_rows.append(_qa_row(
    "Group-day capacity respected",
    passed=(len(viol_group_day) == 0),
    metric=f"viol_group_days={len(viol_group_day)}",
    detail=None if len(viol_group_day) == 0 else "See viol_group_day"
))

# T4: onsite coupling by group-cluster-day
onsite_by_gcd = (
    w_sol_on.loc[w_sol_on["onsite"] == True]
    .groupby(["group_id", "cluster_id", "day"], as_index=False)["w_int"]
    .sum()
    .rename(columns={"w_int": "onsite_workers"})
)

x_sol_on_lookup = x_sol_on[["group_id", "cluster_id", "x_int"]].copy()
viol_onsite_couple = onsite_by_gcd.merge(
    x_sol_on_lookup,
    on=["group_id", "cluster_id"],
    how="left"
)
viol_onsite_couple["x_int"] = viol_onsite_couple["x_int"].fillna(0).astype(int)
viol_onsite_couple = viol_onsite_couple.loc[viol_onsite_couple["onsite_workers"] > viol_onsite_couple["x_int"]]
qa_rows.append(_qa_row(
    "Onsite coupling by group-cluster-day respected",
    passed=(len(viol_onsite_couple) == 0),
    metric=f"viol_rows={len(viol_onsite_couple)}",
    detail=None if len(viol_onsite_couple) == 0 else "See viol_onsite_couple"
))

# T5: no fake deployments
onsite_days_by_gc = (
    w_sol_on.loc[w_sol_on["onsite"] == True]
    .groupby(["group_id", "cluster_id"], as_index=False)["w_int"]
    .sum()
    .rename(columns={"w_int": "onsite_worker_days"})
)

fake_check = x_sol_on.merge(
    onsite_days_by_gc,
    on=["group_id", "cluster_id"],
    how="left"
)
fake_check["onsite_worker_days"] = fake_check["onsite_worker_days"].fillna(0).astype(int)
viol_fake = fake_check.loc[fake_check["onsite_worker_days"] < fake_check["x_int"]]
qa_rows.append(_qa_row(
    "No fake group deployments",
    passed=(len(viol_fake) == 0),
    metric=f"viol_pairs={len(viol_fake)}",
    detail=None if len(viol_fake) == 0 else "See viol_fake"
))

# T6: completion logic
tol = 1e-6
viol_complete = claim_qa.loc[
    (claim_qa["z_bin"] == 1) & (claim_qa["total_progress"] < 1.0 - tol),
    ["claim_id", "z_bin", "total_progress", "severity", "onsite", "online", "due_day"]
]
qa_rows.append(_qa_row(
    "Completion correctness: z=1 implies progress>=1",
    passed=(len(viol_complete) == 0),
    metric=f"viol_claims={len(viol_complete)}",
    detail=None if len(viol_complete) == 0 else "See viol_complete"
))

# T7: incomplete sanity
viol_incomplete = claim_qa.loc[
    (claim_qa["z_bin"] == 0) & (claim_qa["total_progress"] >= 1.0 - tol),
    ["claim_id", "z_bin", "total_progress", "severity", "onsite", "online", "due_day"]
]
qa_rows.append(_qa_row(
    "Sanity: z=0 implies progress<1",
    passed=(len(viol_incomplete) == 0),
    metric=f"viol_claims={len(viol_incomplete)}",
    detail=None if len(viol_incomplete) == 0 else "See viol_incomplete"
))

# T8: team cap per claim/day
MAX_TEAM = 3
viol_team_cap = team_by_claim_day.loc[team_by_claim_day["team_size"] > MAX_TEAM]
qa_rows.append(_qa_row(
    "Team cap per claim/day respected",
    passed=(len(viol_team_cap) == 0),
    metric=f"viol_claim_days={len(viol_team_cap)}",
    detail=None if len(viol_team_cap) == 0 else "See viol_team_cap"
))

# ------------------------------------------------------------
# 4) business inspection
# ------------------------------------------------------------
overall_completed = claim_qa["z_bin"].mean()

by_sev = (
    claim_qa.groupby("severity")["z_bin"]
    .mean()
    .sort_index()
    .reset_index(name="completion_rate")
)

by_mode = (
    claim_qa.groupby(["onsite", "online"])["z_bin"]
    .mean()
    .reset_index(name="completion_rate")
)

deploy_by_cluster = (
    x_sol_on.groupby("cluster_id", as_index=False)["x_int"]
    .sum()
    .rename(columns={"x_int": "n_deployed"})
    .sort_values("n_deployed", ascending=False)
)

team_stats = team_by_claim_day["team_size"].describe()

qa_rows.append(_qa_row(
    "Business: completion rate overall",
    passed=(overall_completed > 0),
    metric=f"{overall_completed:.1%}",
    detail="See by_sev / by_mode"
))

# ------------------------------------------------------------
# 5) present
# ------------------------------------------------------------
qa_summary_grouped = pd.DataFrame(qa_rows)
display(qa_summary_grouped)

print("\n--- GROUPED QA DETAILS (only if failures) ---")
if len(viol_window): display(viol_window.head(50))
if len(viol_group_deploy): display(viol_group_deploy.head(50))
if len(viol_group_day): display(viol_group_day.head(50))
if len(viol_onsite_couple): display(viol_onsite_couple.head(50))
if len(viol_fake): display(viol_fake.head(50))
if len(viol_complete): display(viol_complete.head(50))
if len(viol_incomplete): display(viol_incomplete.head(50))
if len(viol_team_cap): display(viol_team_cap.head(50))

print("\nCompletion rate by severity:")
display(by_sev)

print("\nCompletion rate by mode:")
display(by_mode)

print("\nDeployments by cluster:")
display(deploy_by_cluster.head(20))

print("\nTeam size stats by claim-day:")
display(pd.DataFrame(team_stats).T)

,Test,Status,Metric,Detail
0,Solution exists (SolCount > 0),PASS ✅,SolCount=10,None
1,Grouped work window respected,PASS ✅,viol_rows=0,None
2,Group deployment count <= group size,PASS ✅,viol_groups=0,None
3,Group-day capacity respected,PASS ✅,viol_group_days=0,None
4,Onsite coupling by group-cluster-day respected,PASS ✅,viol_rows=0,None
5,No fake group deployments,PASS ✅,viol_pairs=0,None
6,Completion correctness: z=1 implies progress>=1,PASS ✅,viol_claims=0,None
7,Sanity: z=0 implies progress<1,PASS ✅,viol_claims=0,None
8,Team cap per claim/day respected,PASS ✅,viol_claim_days=0,None
9,Business: completion rate overall,PASS ✅,98.6%,See by_sev / by_mode



--- GROUPED QA DETAILS (only if failures) ---

Completion rate by severity:


,severity,completion_rate
0,1,1.000000
1,2,1.000000
2,3,1.000000
3,4,1.000000
4,5,0.931818



Completion rate by mode:


,onsite,online,completion_rate
0,False,True,1.000000
1,True,False,0.982122



Deployments by cluster:


,cluster_id,n_deployed
7,15,12
1,1,11
26,8,10
24,6,10
5,13,9
25,7,9
21,3,8
0,0,8
2,10,7
12,2,7



Team size stats by claim-day:


,count,mean,std,min,25%,50%,75%,max
team_size,1681.0,1.449137,0.671663,1.0,1.0,1.0,2.0,3.0


In [8]:
# ============================================================
# PERSON-LEVEL QA AFTER MATERIALIZATION
# Assumes:
#   real_deployments_df
#   real_work_assignments_df
#   claims_model_ids
#   adj_group_base
# ============================================================

import pandas as pd

claims_meta = claims_model_ids.copy()
claims_meta["claim_id"] = claims_meta["claim_id"].astype(str)
claims_meta["cluster_id"] = claims_meta["cluster_id"].astype(str)
claims_meta["arrival_day"] = claims_meta["arrival_day"].astype(int)
claims_meta["due_day"] = claims_meta["due_day"].astype(int)
claims_meta["onsite"] = claims_meta["onsite"].astype(bool)
claims_meta["online"] = claims_meta["online"].astype(bool)

rw = real_work_assignments_df.merge(
    claims_meta[["claim_id", "cluster_id", "arrival_day", "due_day", "onsite", "online", "severity"]],
    on="claim_id",
    how="left",
    suffixes=("", "_claim"),
    validate="many_to_one"
)

qa_rows = []

# one cluster per actual adjuster
deploys_per_adj = (
    real_deployments_df.groupby("adjuster_id")["cluster_id"]
    .nunique()
    .reset_index(name="n_clusters")
)
viol_multi_cluster = deploys_per_adj.loc[deploys_per_adj["n_clusters"] > 1]
qa_rows.append({
    "Test": "Actual adjuster deployed to at most one cluster",
    "Status": "PASS ✅" if len(viol_multi_cluster) == 0 else "FAIL ❌",
    "Metric": f"viol_adjusters={len(viol_multi_cluster)}"
})

# no double-booking per adjuster-day
work_per_ad = (
    rw.groupby(["adjuster_id", "day"])
    .size()
    .reset_index(name="n_claims")
)
viol_day_split = work_per_ad.loc[work_per_ad["n_claims"] > 1]
qa_rows.append({
    "Test": "No double-booking per actual adjuster-day",
    "Status": "PASS ✅" if len(viol_day_split) == 0 else "FAIL ❌",
    "Metric": f"viol_adjuster_days={len(viol_day_split)}"
})

# onsite work must match deployed cluster
deploy_lookup = real_deployments_df.rename(columns={"cluster_id": "deployed_cluster"})
onsite_check = rw.loc[rw["onsite"] == True].merge(
    deploy_lookup[["adjuster_id", "deployed_cluster"]],
    on="adjuster_id",
    how="left"
)
viol_onsite_cluster = onsite_check.loc[onsite_check["cluster_id_claim"] != onsite_check["deployed_cluster"]]
qa_rows.append({
    "Test": "Onsite work uses actual deployed cluster",
    "Status": "PASS ✅" if len(viol_onsite_cluster) == 0 else "FAIL ❌",
    "Metric": f"viol_rows={len(viol_onsite_cluster)}"
})

# work window
viol_window = rw.loc[
    (rw["day"] < rw["arrival_day"]) | (rw["day"] > rw["due_day"])
]
qa_rows.append({
    "Test": "Actual work stays within claim window",
    "Status": "PASS ✅" if len(viol_window) == 0 else "FAIL ❌",
    "Metric": f"viol_rows={len(viol_window)}"
})

qa_summary_person = pd.DataFrame(qa_rows)
display(qa_summary_person)

if len(viol_multi_cluster): display(viol_multi_cluster.head(20))
if len(viol_day_split): display(viol_day_split.head(20))
if len(viol_onsite_cluster): display(viol_onsite_cluster.head(20))
if len(viol_window): display(viol_window.head(20))

,Test,Status,Metric
0,Actual adjuster deployed to at most one cluster,PASS ✅,viol_adjusters=0
1,No double-booking per actual adjuster-day,PASS ✅,viol_adjuster_days=0
2,Onsite work uses actual deployed cluster,PASS ✅,viol_rows=0
3,Actual work stays within claim window,PASS ✅,viol_rows=0


## Dataset Creation

In [9]:
# ============================================================
# FINAL OUTPUT TABLES FOR DASHBOARD
# Uses the FINAL MATERIALIZED solution:
#   real_work_assignments_df
#   real_deployments_df
# and preserves the same dashboard-facing structure:
#   assignments_df
#   deployments_df
#   completed_df
#   claims_df
#   adjusters_df
#   adjuster_day_df
#   clusters_df
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0) TYPE STANDARDIZATION
# ------------------------------------------------------------
claims_model_ids["claim_id"] = claims_model_ids["claim_id"].astype(str)
claim_cluster_map["claim_id"] = claim_cluster_map["claim_id"].astype(str)
claim_cluster_map["cluster_id"] = claim_cluster_map["cluster_id"].astype(str)
cluster_summary["cluster_id"] = cluster_summary["cluster_id"].astype(str)

adjusters_core["adjuster_id"] = adjusters_core["adjuster_id"].astype(str)
adj_group_base["adjuster_id"] = adj_group_base["adjuster_id"].astype(str)
adj_group_base["group_id"] = adj_group_base["group_id"].astype(str)

claims_calendar["claim_id"] = claims_calendar["claim_id"].astype(str)
claims_calendar["onsite"] = claims_calendar["onsite"].astype(bool)
claims_calendar["online"] = claims_calendar["online"].astype(bool)

real_work_assignments_df["adjuster_id"] = real_work_assignments_df["adjuster_id"].astype(str)
real_work_assignments_df["group_id"] = real_work_assignments_df["group_id"].astype(str)
real_work_assignments_df["claim_id"] = real_work_assignments_df["claim_id"].astype(str)
real_work_assignments_df["cluster_id"] = real_work_assignments_df["cluster_id"].astype(str)
real_work_assignments_df["day"] = real_work_assignments_df["day"].astype(int)

real_deployments_df["adjuster_id"] = real_deployments_df["adjuster_id"].astype(str)
real_deployments_df["group_id"] = real_deployments_df["group_id"].astype(str)
real_deployments_df["cluster_id"] = real_deployments_df["cluster_id"].astype(str)

# ------------------------------------------------------------
# 1) ASSIGNMENTS TABLE
# same shape idea as before, but based on materialized rows
# ------------------------------------------------------------
assignments_df = real_work_assignments_df.copy()

# merge claim-day metadata from acd2 using real adjuster_id, claim_id, day
assignments_df = assignments_df.merge(
    acd2[[
        "adjuster_id", "claim_id", "day",
        "cluster_id",
        "claim_type", "severity",
        "skill_for_claim",
        "service_days",
        "drive_seconds", "drive_miles",
        "travel_hours", "travel_mult",
        "base_progress_per_day", "progress_per_day",
        "onsite", "online",
        "arrival_day", "due_day"
    ]].drop_duplicates(),
    on=["adjuster_id", "claim_id", "day"],
    how="left",
    suffixes=("", "_acd2")
)

# if cluster_id from materialization exists, keep it as source of truth
if "cluster_id_acd2" in assignments_df.columns:
    assignments_df["cluster_id"] = assignments_df["cluster_id"].fillna(assignments_df["cluster_id_acd2"])
    assignments_df = assignments_df.drop(columns=["cluster_id_acd2"])

# merge adjuster info
assignments_df = assignments_df.merge(
    adjusters_core[[
        "adjuster_id",
        "skill_business", "skill_personal",
        "will_travel",
        "home_city", "home_state",
        "home_lat", "home_lon"
    ]],
    on="adjuster_id",
    how="left"
)

# merge group info for traceability
assignments_df = assignments_df.merge(
    adj_group_base[["adjuster_id", "group_id"]].drop_duplicates(),
    on="adjuster_id",
    how="left",
    suffixes=("", "_grp")
)

# derived fields
assignments_df["modality"] = np.where(assignments_df["onsite"], "onsite", "online")
assignments_df["productivity_loss_pct"] = 1 - assignments_df["travel_mult"]
assignments_df["productivity_loss_abs"] = (
    assignments_df["base_progress_per_day"] - assignments_df["progress_per_day"]
)

assignments_df = assignments_df.rename(columns={
    "progress_per_day": "effective_progress_per_day"
})

# order columns nicely
assignment_cols_preferred = [
    "adjuster_id", "group_id", "claim_id", "day", "cluster_id",
    "claim_type", "severity", "skill_for_claim",
    "service_days",
    "drive_seconds", "drive_miles", "travel_hours",
    "travel_mult", "base_progress_per_day", "effective_progress_per_day",
    "productivity_loss_pct", "productivity_loss_abs",
    "onsite", "online", "modality",
    "arrival_day", "due_day",
    "skill_business", "skill_personal", "will_travel",
    "home_city", "home_state", "home_lat", "home_lon"
]
assignment_cols_existing = [c for c in assignment_cols_preferred if c in assignments_df.columns]
assignments_df = assignments_df[assignment_cols_existing + [c for c in assignments_df.columns if c not in assignment_cols_existing]]

# ------------------------------------------------------------
# 2) DEPLOYMENTS TABLE
# ------------------------------------------------------------
deployments_df = real_deployments_df.copy()

deployments_df = deployments_df.merge(
    adjusters_core[[
        "adjuster_id",
        "skill_business", "skill_personal",
        "will_travel",
        "home_city", "home_state",
        "home_lat", "home_lon"
    ]],
    on="adjuster_id",
    how="left"
)

deployments_df = deployments_df.merge(
    cluster_summary[[
        "cluster_id", "claims", "onsite_claims", "online_claims", "has_onsite"
    ]],
    on="cluster_id",
    how="left"
)

# ------------------------------------------------------------
# 3) COMPLETED / CLAIM COMPLETION TABLE
# ------------------------------------------------------------
completed_df = pd.DataFrame({
    "claim_id": [str(i) for i in I],
    "completed": [bool(z[i].X > 0.5) for i in I]
})

# work summary per claim
claim_work_summary = (
    assignments_df.groupby("claim_id", as_index=False)
    .agg(
        first_work_day=("day", "min"),
        last_work_day=("day", "max"),
        total_workdays=("day", "count"),
        n_adjusters_used=("adjuster_id", "nunique"),
        total_effective_progress=("effective_progress_per_day", "sum"),
        total_base_progress=("base_progress_per_day", "sum"),
        total_productivity_loss=("productivity_loss_abs", "sum"),
        total_travel_hours=("travel_hours", "sum"),
        total_drive_miles=("drive_miles", "sum")
    )
)

completed_df = completed_df.merge(
    claims_model_ids[[
        "claim_id", "severity", "arrival_day", "due_day",
        "onsite", "online", "cluster_id", "w"
    ]],
    on="claim_id",
    how="left"
)

completed_df = completed_df.merge(
    claim_work_summary,
    on="claim_id",
    how="left"
)

completed_df["completion_day"] = np.where(
    completed_df["completed"],
    completed_df["last_work_day"],
    np.nan
)

completed_df["days_to_complete"] = np.where(
    completed_df["completed"],
    completed_df["completion_day"] - completed_df["arrival_day"] + 1,
    np.nan
)

completed_df["cycle_time_days_opt"] = np.where(
    completed_df["completed"],
    completed_df["completion_day"] - completed_df["arrival_day"],
    np.nan
)

completed_df["completed_on_time"] = np.where(
    completed_df["completed"] &
    (completed_df["completion_day"] <= completed_df["due_day"]),
    True,
    False
)

# ------------------------------------------------------------
# 4) CLAIMS MASTER
# ------------------------------------------------------------
claims_df = claims_calendar.copy()
claims_df["claim_id"] = claims_df["claim_id"].astype(str)

claims_df = claims_df.merge(
    claim_cluster_map,
    on="claim_id",
    how="left"
)

claims_df = claims_df.merge(
    completed_df[[
        "claim_id", "completed", "completion_day",
        "days_to_complete", "cycle_time_days_opt",
        "completed_on_time", "total_workdays",
        "n_adjusters_used", "total_effective_progress",
        "total_travel_hours", "total_drive_miles"
    ]],
    on="claim_id",
    how="left"
)

# ------------------------------------------------------------
# 5) ADJUSTERS MASTER
# ------------------------------------------------------------
adjusters_df = adjusters_core.copy()
adjusters_df["adjuster_id"] = adjusters_df["adjuster_id"].astype(str)

adjusters_df = adjusters_df.merge(
    adj_group_base[["adjuster_id", "group_id"]].drop_duplicates(),
    on="adjuster_id",
    how="left"
)

# deployed flag
deployed_adjusters = deployments_df[["adjuster_id"]].drop_duplicates().assign(deployed_flag=1)
adjusters_df = adjusters_df.merge(
    deployed_adjusters,
    on="adjuster_id",
    how="left"
)
adjusters_df["deployed_flag"] = adjusters_df["deployed_flag"].fillna(0).astype(int)

# ------------------------------------------------------------
# 6) ADJUSTER-DAY TABLE
# recreate from planning horizon + actual adjusters
# ------------------------------------------------------------
adjuster_day_df = (
    adjusters_df[["adjuster_id"]]
    .assign(key=1)
    .merge(planning_days.assign(key=1), on="key")
    .drop(columns="key")
)

adjuster_day_df["adjuster_id"] = adjuster_day_df["adjuster_id"].astype(str)
adjuster_day_df["day"] = adjuster_day_df["day"].astype(int)
adjuster_day_df["capacity_days"] = 1.0

adjuster_day_df = adjuster_day_df.merge(
    assignments_df[[
        "adjuster_id", "day", "claim_id", "cluster_id",
        "modality", "effective_progress_per_day",
        "productivity_loss_abs", "travel_hours", "drive_miles"
    ]],
    on=["adjuster_id", "day"],
    how="left"
)

adjuster_day_df["assigned_flag"] = adjuster_day_df["claim_id"].notna().astype(int)

# add deployment cluster for reference
adjuster_day_df = adjuster_day_df.merge(
    deployments_df[["adjuster_id", "cluster_id"]].drop_duplicates().rename(columns={"cluster_id": "deployed_cluster"}),
    on="adjuster_id",
    how="left"
)

# ------------------------------------------------------------
# 7) CLUSTERS TABLE
# ------------------------------------------------------------
clusters_df = cluster_summary.copy()
clusters_df["cluster_id"] = clusters_df["cluster_id"].astype(str)

cluster_deploy_summary = (
    deployments_df.groupby("cluster_id", as_index=False)["adjuster_id"]
    .nunique()
    .rename(columns={"adjuster_id": "n_deployed_adjusters"})
)

cluster_assignment_summary = (
    assignments_df.groupby("cluster_id", as_index=False)
    .agg(
        n_assignment_rows=("claim_id", "count"),
        n_claims_worked=("claim_id", "nunique"),
        n_adjusters_working=("adjuster_id", "nunique")
    )
)

clusters_df = clusters_df.merge(cluster_deploy_summary, on="cluster_id", how="left")
clusters_df = clusters_df.merge(cluster_assignment_summary, on="cluster_id", how="left")

for c in ["n_deployed_adjusters", "n_assignment_rows", "n_claims_worked", "n_adjusters_working"]:
    if c in clusters_df.columns:
        clusters_df[c] = clusters_df[c].fillna(0)


# ============================================================
# POWER BI COMPATIBLE EXPORTS
# Exports FINAL materialized solution with EXACT legacy schemas
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) ASSIGNMENTS  -> EXACT 28 COLUMNS
# ------------------------------------------------------------
assignments_export = assignments_df.copy()

for col in [
    "adjuster_id", "claim_id", "day", "cluster_id", "claim_type", "severity",
    "skill_for_claim", "service_days", "drive_seconds", "drive_miles",
    "travel_hours", "travel_mult", "base_progress_per_day",
    "effective_progress_per_day", "onsite", "online", "arrival_day", "due_day",
    "skill_business", "skill_personal", "will_travel", "home_city",
    "home_state", "home_lat", "home_lon", "modality",
    "productivity_loss_pct", "productivity_loss_abs"
]:
    if col not in assignments_export.columns:
        assignments_export[col] = np.nan

assignments_export = assignments_export[[
    "adjuster_id", "claim_id", "day", "cluster_id", "claim_type", "severity",
    "skill_for_claim", "service_days", "drive_seconds", "drive_miles",
    "travel_hours", "travel_mult", "base_progress_per_day",
    "effective_progress_per_day", "onsite", "online", "arrival_day", "due_day",
    "skill_business", "skill_personal", "will_travel", "home_city",
    "home_state", "home_lat", "home_lon", "modality",
    "productivity_loss_pct", "productivity_loss_abs"
]].copy()

# ------------------------------------------------------------
# 2) DEPLOYMENTS -> EXACT 13 COLUMNS
# ------------------------------------------------------------
deployments_export = deployments_df.copy()

for col in [
    "adjuster_id", "cluster_id", "skill_business", "skill_personal",
    "will_travel", "home_city", "home_state", "home_lat", "home_lon",
    "claims", "onsite_claims", "online_claims", "has_onsite"
]:
    if col not in deployments_export.columns:
        deployments_export[col] = np.nan

deployments_export = deployments_export[[
    "adjuster_id", "cluster_id", "skill_business", "skill_personal",
    "will_travel", "home_city", "home_state", "home_lat", "home_lon",
    "claims", "onsite_claims", "online_claims", "has_onsite"
]].copy()

# ------------------------------------------------------------
# 3) CLAIM COMPLETION -> EXACT 19 COLUMNS
# ------------------------------------------------------------
claim_completion_export = completed_df.copy()

for col in [
    "claim_id", "completed", "severity", "arrival_day", "due_day",
    "onsite", "online", "cluster_id", "w", "first_work_day",
    "last_work_day", "total_workdays", "n_adjusters_used",
    "total_effective_progress", "total_base_progress",
    "total_productivity_loss", "completion_day",
    "days_to_complete", "completed_on_time"
]:
    if col not in claim_completion_export.columns:
        claim_completion_export[col] = np.nan

# make sure old field name exists
if "total_effective_progress" not in claim_completion_export.columns and "total_progress" in claim_completion_export.columns:
    claim_completion_export["total_effective_progress"] = claim_completion_export["total_progress"]

claim_completion_export = claim_completion_export[[
    "claim_id", "completed", "severity", "arrival_day", "due_day",
    "onsite", "online", "cluster_id", "w", "first_work_day",
    "last_work_day", "total_workdays", "n_adjusters_used",
    "total_effective_progress", "total_base_progress",
    "total_productivity_loss", "completion_day",
    "days_to_complete", "completed_on_time"
]].copy()

# ------------------------------------------------------------
# 4) CLAIMS MASTER -> EXACT 20 COLUMNS
# ------------------------------------------------------------
claims_master_export = claims_df.copy()

for col in [
    "claim_id", "claim_type", "severity", "arrival_date", "arrival_day",
    "due_date", "due_day", "sla_days", "onsite", "online",
    "lat", "lon", "cluster_id", "completed", "completion_day",
    "days_to_complete", "completed_on_time", "total_workdays",
    "n_adjusters_used", "total_effective_progress"
]:
    if col not in claims_master_export.columns:
        claims_master_export[col] = np.nan

claims_master_export = claims_master_export[[
    "claim_id", "claim_type", "severity", "arrival_date", "arrival_day",
    "due_date", "due_day", "sla_days", "onsite", "online",
    "lat", "lon", "cluster_id", "completed", "completion_day",
    "days_to_complete", "completed_on_time", "total_workdays",
    "n_adjusters_used", "total_effective_progress"
]].copy()

# ------------------------------------------------------------
# 5) ADJUSTERS MASTER -> EXACT 9 COLUMNS
# ------------------------------------------------------------
adjusters_master_export = adjusters_df.copy()

for col in [
    "adjuster_id", "skill_business", "skill_personal", "will_travel",
    "home_city", "home_state", "home_lat", "home_lon", "shift_capacity_days"
]:
    if col not in adjusters_master_export.columns:
        adjusters_master_export[col] = np.nan

adjusters_master_export = adjusters_master_export[[
    "adjuster_id", "skill_business", "skill_personal", "will_travel",
    "home_city", "home_state", "home_lat", "home_lon", "shift_capacity_days"
]].copy()

# ------------------------------------------------------------
# 6) ADJUSTER DAY -> EXACT 8 COLUMNS
# ------------------------------------------------------------
adjuster_day_export = adjuster_day_df.copy()

for col in [
    "adjuster_id", "capacity_days", "day", "claim_id",
    "modality", "effective_progress_per_day",
    "productivity_loss_abs", "assigned_flag"
]:
    if col not in adjuster_day_export.columns:
        adjuster_day_export[col] = np.nan

adjuster_day_export = adjuster_day_export[[
    "adjuster_id", "capacity_days", "day", "claim_id",
    "modality", "effective_progress_per_day",
    "productivity_loss_abs", "assigned_flag"
]].copy()

# ------------------------------------------------------------
# 7) CLUSTERS -> keep as current
# ------------------------------------------------------------
clusters_export = clusters_df.copy()

# ============================================================
# BUILD HUMAN-READABLE CLUSTER LABELS
# - creates cluster_name and cluster_abbrev
# - uses dominant city/state if available
# - otherwise falls back to centroid-based generic names
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) Start from claims + cluster assignment
# ------------------------------------------------------------
cluster_label_df = claims_df.copy()
cluster_label_df["cluster_id"] = cluster_label_df["cluster_id"].astype(str)

# candidate location columns if they exist in your original cleaned claims file
candidate_city_cols = [c for c in [
    "Accident City", "City", "city", "accident_city"
] if c in ds_claim_optClean.columns]

candidate_state_cols = [c for c in [
    "Accident State", "State", "state", "accident_state"
] if c in ds_claim_optClean.columns]

# attach city/state if available
claims_loc = ds_claim_optClean.copy()
if "Claim Number" in claims_loc.columns:
    claims_loc["claim_id"] = claims_loc["Claim Number"].astype(str)
else:
    claims_loc["claim_id"] = claims_loc["claim_id"].astype(str)

loc_keep = ["claim_id"]
if candidate_city_cols:
    loc_keep.append(candidate_city_cols[0])
if candidate_state_cols:
    loc_keep.append(candidate_state_cols[0])

claims_loc = claims_loc[loc_keep].copy()

cluster_label_df["claim_id"] = cluster_label_df["claim_id"].astype(str)
cluster_label_df = cluster_label_df.merge(
    claims_loc,
    on="claim_id",
    how="left"
)

city_col = candidate_city_cols[0] if candidate_city_cols else None
state_col = candidate_state_cols[0] if candidate_state_cols else None

# ------------------------------------------------------------
# 2) Cluster centroids
# ------------------------------------------------------------
cluster_centroids = (
    cluster_label_df.groupby("cluster_id", as_index=False)
    .agg(
        centroid_lat=("lat", "mean"),
        centroid_lon=("lon", "mean"),
        n_claims=("claim_id", "nunique")
    )
)

# ------------------------------------------------------------
# 3) Dominant city/state by cluster
# ------------------------------------------------------------
def mode_or_null(series):
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    if len(s) == 0:
        return None
    vc = s.value_counts()
    return vc.index[0]

agg_dict = {}
if city_col:
    agg_dict["dominant_city"] = (city_col, mode_or_null)
if state_col:
    agg_dict["dominant_state"] = (state_col, mode_or_null)

if agg_dict:
    cluster_dom = cluster_label_df.groupby("cluster_id", as_index=False).agg(**agg_dict)
else:
    cluster_dom = pd.DataFrame({"cluster_id": cluster_centroids["cluster_id"]})

cluster_labels = cluster_centroids.merge(cluster_dom, on="cluster_id", how="left")

# ------------------------------------------------------------
# 4) Build a base label
# preference:
#   city + state
#   city only
#   state only
#   generic cluster
# ------------------------------------------------------------
def build_base_label(row):
    city = row.get("dominant_city", None)
    state = row.get("dominant_state", None)

    if pd.notna(city) and city:
        return str(city).strip()
    if pd.notna(state) and state:
        return str(state).strip()
    return f"Cluster {row['cluster_id']}"

cluster_labels["base_label"] = cluster_labels.apply(build_base_label, axis=1)

# ------------------------------------------------------------
# 5) If multiple clusters share same base label,
# add directional suffix based on centroid position
# ------------------------------------------------------------
def assign_directional_names(df):
    if len(df) == 1:
        df["cluster_name"] = df["base_label"]
        return df

    # choose north/south if spread in latitude is stronger than longitude
    lat_span = df["centroid_lat"].max() - df["centroid_lat"].min()
    lon_span = df["centroid_lon"].max() - df["centroid_lon"].min()

    df = df.copy()

    if lat_span >= lon_span:
        df = df.sort_values("centroid_lat", ascending=False).reset_index(drop=True)
        if len(df) == 2:
            suffixes = ["North", "South"]
        elif len(df) == 3:
            suffixes = ["North", "Central", "South"]
        else:
            suffixes = [f"N{i+1}" for i in range(len(df))]
    else:
        df = df.sort_values("centroid_lon", ascending=False).reset_index(drop=True)
        if len(df) == 2:
            suffixes = ["East", "West"]
        elif len(df) == 3:
            suffixes = ["East", "Central", "West"]
        else:
            suffixes = [f"E{i+1}" for i in range(len(df))]

    df["cluster_name"] = [
        f"{base} {suf}" for base, suf in zip(df["base_label"], suffixes)
    ]
    return df

cluster_labels = (
    cluster_labels.groupby("base_label", group_keys=False)
    .apply(assign_directional_names)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6) Abbreviation
# ------------------------------------------------------------
def abbrev(text, max_parts=2, part_len=3):
    parts = str(text).replace("-", " ").split()
    parts = [p[:part_len].upper() for p in parts[:max_parts]]
    return "-".join(parts)

cluster_labels["cluster_abbrev"] = cluster_labels["cluster_name"].apply(abbrev)

# optional: make abbrevs unique if collisions exist
dup_counts = cluster_labels["cluster_abbrev"].value_counts()
dups = dup_counts[dup_counts > 1].index.tolist()

if dups:
    cluster_labels["cluster_abbrev"] = np.where(
        cluster_labels["cluster_abbrev"].isin(dups),
        cluster_labels["cluster_abbrev"] + "-" + cluster_labels["cluster_id"].astype(str),
        cluster_labels["cluster_abbrev"]
    )

# ------------------------------------------------------------
# 7) Final mapping table
# ------------------------------------------------------------
cluster_name_map = cluster_labels[[
    "cluster_id",
    "cluster_name",
    "cluster_abbrev",
    "base_label",
    "dominant_city" if "dominant_city" in cluster_labels.columns else "cluster_id",
    "dominant_state" if "dominant_state" in cluster_labels.columns else "cluster_id",
    "centroid_lat",
    "centroid_lon",
    "n_claims"
]].copy()

# clean accidental duplicate placeholder cols
cluster_name_map = cluster_name_map.loc[:, ~cluster_name_map.columns.duplicated()]

print("Cluster label suggestions:")
display(cluster_name_map.sort_values("cluster_id"))

# save so you can review/edit manually if needed
cluster_name_map.to_csv(f"{output_path}cluster_name_map.csv", index=False)
print(f"Saved: {output_path}cluster_name_map.csv")

# ============================================================
# MERGE CLUSTER LABELS INTO OUTPUT TABLES
# ============================================================

for df_name in ["clusters_df", "claims_df", "deployments_df", "assignments_df", "completed_df"]:
    if df_name in globals():
        df_obj = globals()[df_name]
        if "cluster_id" in df_obj.columns:
            df_obj["cluster_id"] = df_obj["cluster_id"].astype(str)
            df_obj = df_obj.merge(
                cluster_name_map[["cluster_id", "cluster_name", "cluster_abbrev"]],
                on="cluster_id",
                how="left"
            )
            globals()[df_name] = df_obj

print("Cluster labels merged into output tables.")

# ------------------------------------------------------------
# 8) SAVE EXACT FILES FOR POWER BI
# ------------------------------------------------------------
assignments_export.to_csv(f"{output_path}assignments.csv", index=False)
deployments_export.to_csv(f"{output_path}deployments.csv", index=False)
claim_completion_export.to_csv(f"{output_path}claim_completion.csv", index=False)
claims_master_export.to_csv(f"{output_path}claims_master.csv", index=False)
adjusters_master_export.to_csv(f"{output_path}adjusters_master.csv", index=False)
adjuster_day_export.to_csv(f"{output_path}adjuster_day.csv", index=False)
clusters_export.to_csv(f"{output_path}clusters.csv", index=False)

print("Power BI compatible files saved.")

Cluster label suggestions:


/tmp/ipykernel_3662407/3350878299.py:612: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_directional_names)


,cluster_id,cluster_name,cluster_abbrev,base_label,centroid_lat,centroid_lon,n_claims
0,0,Cluster 0,CLU-0,Cluster 0,36.319737,-86.574784,202
1,1,Cluster 1,CLU-1,Cluster 1,40.162582,-75.209792,72
2,10,Cluster 10,CLU-10,Cluster 10,40.732388,-73.976678,49
3,11,Cluster 11,CLU-11,Cluster 11,31.552062,-82.166125,8
4,12,Cluster 12,CLU-12,Cluster 12,35.930782,-84.007100,11
5,13,Cluster 13,CLU-13,Cluster 13,39.317740,-76.580353,47
6,14,Cluster 14,CLU-14,Cluster 14,32.688274,-90.997011,19
7,15,Cluster 15,CLU-15,Cluster 15,36.568124,-87.406971,134
8,16,Cluster 16,CLU-16,Cluster 16,43.072550,-78.104850,2
9,17,Cluster 17,CLU-17,Cluster 17,42.190086,-71.308132,28


Saved: Datasets/Outputs/cluster_name_map.csv
Cluster labels merged into output tables.
Power BI compatible files saved.


In [19]:
# ============================================================
# CLUSTER LABELS FROM CENTROID LAT/LON -> STATE-NUMBER
# Example: GA-1, GA-2, FL-1
# ============================================================

import pandas as pd
import geopandas as gpd

# Use your existing clusters_df
clusters_df = clusters_df.copy()
clusters_df["cluster_id"] = clusters_df["cluster_id"].astype(str)

# Adjust these if your columns have different names
lat_col = "centroid_lat" if "centroid_lat" in clusters_df.columns else "lat"
lon_col = "centroid_lon" if "centroid_lon" in clusters_df.columns else "lon"

# Convert clusters to GeoDataFrame
clusters_gdf = gpd.GeoDataFrame(
    clusters_df,
    geometry=gpd.points_from_xy(clusters_df[lon_col], clusters_df[lat_col]),
    crs="EPSG:4326"
)

# US Census state boundaries
states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"
states_gdf = gpd.read_file(states_url)

# Keep only state abbreviation + geometry
states_gdf = states_gdf[["STUSPS", "NAME", "geometry"]].copy()
states_gdf = states_gdf.to_crs("EPSG:4326")

# Spatial join: assign each cluster centroid to a state
clusters_gdf = gpd.sjoin(
    clusters_gdf,
    states_gdf,
    how="left",
    predicate="within"
)

clusters_gdf = clusters_gdf.rename(columns={
    "STUSPS": "cluster_state",
    "NAME": "cluster_state_name"
})

clusters_df = pd.DataFrame(clusters_gdf.drop(columns=["geometry", "index_right"]))

# Fallback if any centroid did not fall inside a state polygon
clusters_df["cluster_state"] = clusters_df["cluster_state"].fillna("UNK")

# Number clusters within each state
clusters_df = clusters_df.sort_values(["cluster_state", "cluster_id"]).copy()
clusters_df["cluster_state_num"] = clusters_df.groupby("cluster_state").cumcount() + 1

# Final labels
clusters_df["cluster_name"] = (
    clusters_df["cluster_state"] + "-" + clusters_df["cluster_state_num"].astype(str)
)

clusters_df["cluster_abbrev"] = clusters_df["cluster_name"]

display(
    clusters_df[[
        "cluster_id",
        lat_col,
        lon_col,
        "cluster_state",
        "cluster_state_name",
        "cluster_name",
        "cluster_abbrev"
    ]]
)

clusters_df.to_csv(f"{output_path}clusters.csv", index=False)

KeyError: 'lon'

In [12]:
# ============================================================
# OPTIMIZATION METRICS SUMMARY (FINAL MATERIALIZED VERSION)
# ============================================================

import pandas as pd
import numpy as np

required_objs = [
    "assignments_df", "deployments_df", "completed_df",
    "adjuster_day_df", "claims_model_ids", "adjusters_df", "m"
]
missing_objs = [x for x in required_objs if x not in globals()]
if missing_objs:
    raise NameError(f"Missing required objects before optimization metrics run: {missing_objs}")

assignments_df = assignments_df.copy()
deployments_df = deployments_df.copy()
completed_df = completed_df.copy()
adjuster_day_df = adjuster_day_df.copy()

if "w" not in completed_df.columns:
    completed_df["w"] = pd.to_numeric(completed_df.get("severity", 1), errors="coerce").fillna(1)

completed_df["completed"] = completed_df["completed"].fillna(False).astype(bool)
if "completed_on_time" not in completed_df.columns:
    completed_df["completed_on_time"] = completed_df["completed"]

n_claims_total = len(completed_df)
n_claims_completed = int(completed_df["completed"].sum())

pct_completed_within_sla = (
    100.0 * completed_df["completed_on_time"].mean()
    if n_claims_total > 0 else np.nan
)

weighted_completed = completed_df.loc[completed_df["completed"], "w"].sum()
weighted_total = completed_df["w"].sum()
weighted_completion_pct = (
    100.0 * weighted_completed / weighted_total
    if weighted_total > 0 else np.nan
)

avg_cycle_time_days = completed_df["cycle_time_days_opt"].mean()
median_cycle_time_days = completed_df["cycle_time_days_opt"].median()

used_adjuster_days = int(adjuster_day_df["assigned_flag"].sum())
total_capacity_days = float(adjuster_day_df["capacity_days"].sum())

adjuster_utilization_pct = (
    100.0 * used_adjuster_days / total_capacity_days
    if total_capacity_days > 0 else np.nan
)

all_adjusters = sorted(adjusters_df["adjuster_id"].astype(str).unique().tolist())

adjuster_days_worked = (
    adjuster_day_df.groupby("adjuster_id", as_index=False)["assigned_flag"]
    .sum()
    .rename(columns={"assigned_flag": "days_worked"})
)

adjuster_days_worked = (
    pd.DataFrame({"adjuster_id": all_adjusters})
    .merge(adjuster_days_worked, on="adjuster_id", how="left")
)
adjuster_days_worked["days_worked"] = adjuster_days_worked["days_worked"].fillna(0)

workload_std_days = adjuster_days_worked["days_worked"].std(ddof=0)
workload_mean_days = adjuster_days_worked["days_worked"].mean()
workload_cv_days = (
    workload_std_days / workload_mean_days
    if workload_mean_days > 0 else np.nan
)

total_travel_hours = float(assignments_df["travel_hours"].fillna(0).sum()) if "travel_hours" in assignments_df.columns else 0.0
total_drive_miles = float(assignments_df["drive_miles"].fillna(0).sum()) if "drive_miles" in assignments_df.columns else 0.0

deployment_count = len(deployments_df)
deployed_adjuster_count = deployments_df["adjuster_id"].nunique() if not deployments_df.empty else 0

total_productivity_loss = float(assignments_df["productivity_loss_abs"].fillna(0).sum()) if "productivity_loss_abs" in assignments_df.columns else 0.0
avg_productivity_loss_pct = (
    100.0 * assignments_df["productivity_loss_pct"].fillna(0).mean()
    if "productivity_loss_pct" in assignments_df.columns and len(assignments_df) > 0 else 0.0
)

allocation_runtime_seconds = float(m.Runtime) if "m" in globals() else np.nan

optimization_metrics = {
    "n_claims_total": n_claims_total,
    "n_claims_completed_within_sla": n_claims_completed,
    "pct_completed_within_sla": pct_completed_within_sla,
    "weighted_completion_pct": weighted_completion_pct,
    "avg_cycle_time_days": avg_cycle_time_days,
    "median_cycle_time_days": median_cycle_time_days,
    "adjuster_utilization_pct": adjuster_utilization_pct,
    "workload_std_days": workload_std_days,
    "workload_cv_days": workload_cv_days,
    "total_travel_hours": total_travel_hours,
    "total_drive_miles": total_drive_miles,
    "deployment_count": deployment_count,
    "deployed_adjuster_count": deployed_adjuster_count,
    "total_productivity_loss": total_productivity_loss,
    "avg_productivity_loss_pct": avg_productivity_loss_pct,
    "allocation_runtime_seconds": allocation_runtime_seconds,
    "used_adjuster_days": used_adjuster_days,
    "total_capacity_days": total_capacity_days,
}

optimization_metrics_df = pd.DataFrame({
    "metric": list(optimization_metrics.keys()),
    "value": list(optimization_metrics.values())
})

optimization_claim_results_df = completed_df.copy()

optimization_adjuster_summary_df = adjuster_days_worked.copy()

if not assignments_df.empty:
    optimization_adjuster_claim_list = (
        assignments_df.groupby(["adjuster_id", "claim_id"], as_index=False)
        .agg(
            days_worked=("day", "count"),
            total_effective_progress=("effective_progress_per_day", "sum"),
            total_travel_hours=("travel_hours", "sum"),
            total_drive_miles=("drive_miles", "sum"),
            total_productivity_loss=("productivity_loss_abs", "sum"),
        )
        .merge(
            completed_df[["claim_id", "severity", "onsite", "online", "cluster_id", "due_day"]],
            on="claim_id",
            how="left"
        )
        .sort_values(["days_worked", "severity"], ascending=[False, False])
        .reset_index(drop=True)
    )
else:
    optimization_adjuster_claim_list = pd.DataFrame(columns=[
        "adjuster_id", "claim_id", "days_worked", "total_effective_progress",
        "total_travel_hours", "total_drive_miles", "total_productivity_loss",
        "severity", "onsite", "online", "cluster_id", "due_day"
    ])

print("==================================================")
print("OPTIMIZATION RESULTS")
print("==================================================")
print(f"Claims completed within SLA: {n_claims_completed} / {n_claims_total} ({pct_completed_within_sla:.2f}%)")
print(f"Weighted completion %     : {weighted_completion_pct:.2f}%")
print(f"Average cycle time (days) : {avg_cycle_time_days:.2f}")
print(f"Median cycle time (days)  : {median_cycle_time_days:.2f}")
print(f"Adjuster utilization %    : {adjuster_utilization_pct:.2f}%")
print(f"Workload CV (days worked) : {workload_cv_days:.4f}")
print(f"Total travel hours        : {total_travel_hours:.2f}")
print(f"Total drive miles         : {total_drive_miles:.2f}")
print(f"Deployment count          : {deployment_count}")
print(f"Runtime (seconds)         : {allocation_runtime_seconds:.4f}")

print("\nTop 20 optimization adjuster-claim assignments:")
print(optimization_adjuster_claim_list.head(20))

OPTIMIZATION RESULTS
Claims completed within SLA: 1038 / 1053 (98.58%)
Weighted completion %     : 97.94%
Average cycle time (days) : 14.90
Median cycle time (days)  : 14.00
Adjuster utilization %    : 6.57%
Workload CV (days worked) : 2.0681
Total travel hours        : 3029.16
Total drive miles         : 116523.25
Deployment count          : 124
Runtime (seconds)         : 342.5830

Top 20 optimization adjuster-claim assignments:
   adjuster_id  claim_id  days_worked  total_effective_progress  \
0     21204478  22920771           14                     1.050   
1     11494435  63187573           12                     0.600   
2     14582357  63187573           12                     0.600   
3     39267987  94141142            8                     0.800   
4     46986893  75125571            8                     1.000   
5     54636222  19727765            8                     0.800   
6     59430808  39620107            8                     1.000   
7     10134858  82908253     

In [11]:
import pandas as pd
from shapely.geometry import MultiPoint, mapping, Polygon
import json
import math

# ============================================================
# CLUSTER GEOMETRIES FROM claims_df
# ============================================================

df = claims_df.copy()
df["cluster_id"] = df["cluster_id"].astype(str)
df = df.dropna(subset=["cluster_id", "lat", "lon"]).copy()

# ------------------------------------------------------------
# 1) Convex hull polygons from claim points
# ------------------------------------------------------------
features = []

for cluster_id, group in df.groupby("cluster_id"):
    points = list(zip(group["lon"], group["lat"]))  # (lon, lat)

    if len(points) < 3:
        continue  # cannot build a meaningful hull polygon

    hull = MultiPoint(points).convex_hull

    feature = {
        "type": "Feature",
        "properties": {
            "cluster_id": str(cluster_id)
        },
        "geometry": mapping(hull)
    }

    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open(f"{output_path}clusters.geojson", "w") as f:
    json.dump(geojson, f)

# ------------------------------------------------------------
# 2) Centroid diamonds for all clusters
# useful when a cluster has too few points for a polygon
# ------------------------------------------------------------
features = []
half_size_m = 5000
earth_radius_m = 6371000.0

for cluster_id, group in df.groupby("cluster_id"):
    points = list(zip(group["lon"], group["lat"]))

    if len(points) == 0:
        continue

    if len(points) == 1:
        centroid_lon, centroid_lat = points[0]
    else:
        hull = MultiPoint(points).convex_hull
        centroid = hull.centroid
        centroid_lon, centroid_lat = centroid.x, centroid.y

    lat_rad = math.radians(centroid_lat)

    dlat = (half_size_m / earth_radius_m) * (180.0 / math.pi)
    dlon = dlat / max(math.cos(lat_rad), 1e-9)

    poly = Polygon([
        (centroid_lon, centroid_lat + dlat),
        (centroid_lon + dlon, centroid_lat),
        (centroid_lon, centroid_lat - dlat),
        (centroid_lon - dlon, centroid_lat),
        (centroid_lon, centroid_lat + dlat)
    ])

    features.append({
        "type": "Feature",
        "properties": {
            "cluster_id": str(cluster_id)
        },
        "geometry": mapping(poly)
    })

centroid_geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open(f"{output_path}cluster_centroids.geojson", "w") as f:
    json.dump(centroid_geojson, f)

print(f"Saved: {output_path}clusters.geojson")
print(f"Saved: {output_path}cluster_centroids.geojson")

Saved: Datasets/Outputs/clusters.geojson
Saved: Datasets/Outputs/cluster_centroids.geojson


In [20]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPoint

# ============================================================
# BUILD CLUSTERS TABLE WITH:
# - centroid lat/lon
# - state from centroid
# - label: STATE-NUMBER
# ============================================================

df = claims_df.copy()
df["cluster_id"] = df["cluster_id"].astype(str)
df = df.dropna(subset=["cluster_id", "lat", "lon"]).copy()

# ------------------------------------------------------------
# 1) Compute centroids per cluster
# ------------------------------------------------------------
centroid_rows = []

for cluster_id, group in df.groupby("cluster_id"):
    points = list(zip(group["lon"], group["lat"]))

    if len(points) == 0:
        continue

    if len(points) == 1:
        centroid_lon, centroid_lat = points[0]
    else:
        hull = MultiPoint(points).convex_hull
        centroid = hull.centroid
        centroid_lon, centroid_lat = centroid.x, centroid.y

    centroid_rows.append({
        "cluster_id": cluster_id,
        "centroid_lat": centroid_lat,
        "centroid_lon": centroid_lon,
        "n_claims": len(points)
    })

clusters_df = pd.DataFrame(centroid_rows)

# ------------------------------------------------------------
# 2) Convert to GeoDataFrame
# ------------------------------------------------------------
clusters_gdf = gpd.GeoDataFrame(
    clusters_df,
    geometry=gpd.points_from_xy(clusters_df["centroid_lon"], clusters_df["centroid_lat"]),
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# 3) Load US states shapefile
# ------------------------------------------------------------
states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"
states_gdf = gpd.read_file(states_url)

states_gdf = states_gdf[["STUSPS", "NAME", "geometry"]].to_crs("EPSG:4326")

# ------------------------------------------------------------
# 4) Spatial join: assign state to each centroid
# ------------------------------------------------------------
clusters_gdf = gpd.sjoin(
    clusters_gdf,
    states_gdf,
    how="left",
    predicate="within"
)

clusters_df = pd.DataFrame(clusters_gdf.drop(columns=["geometry", "index_right"]))

clusters_df = clusters_df.rename(columns={
    "STUSPS": "cluster_state",
    "NAME": "cluster_state_name"
})

clusters_df["cluster_state"] = clusters_df["cluster_state"].fillna("UNK")

# ------------------------------------------------------------
# 5) Create STATE-N numbering
# ------------------------------------------------------------
clusters_df = clusters_df.sort_values(["cluster_state", "cluster_id"]).copy()

clusters_df["cluster_state_num"] = (
    clusters_df.groupby("cluster_state").cumcount() + 1
)

clusters_df["cluster_name"] = (
    clusters_df["cluster_state"] + "-" + clusters_df["cluster_state_num"].astype(str)
)

clusters_df["cluster_abbrev"] = clusters_df["cluster_name"]

# ------------------------------------------------------------
# 6) Save clusters.csv
# ------------------------------------------------------------
clusters_df.to_csv(f"{output_path}clusters.csv", index=False)

print("clusters.csv created with centroid + state labels")
display(clusters_df.head())

clusters.csv created with centroid + state labels


,cluster_id,centroid_lat,centroid_lon,n_claims,cluster_state,cluster_state_name,cluster_state_num,cluster_name,cluster_abbrev
15,22,32.139200,-85.599067,4,AL,Alabama,1,AL-1,AL-1
29,8,33.957570,-86.847302,81,AL,Alabama,2,AL-2,AL-2
27,6,41.606167,-72.742256,60,CT,Connecticut,1,CT-1,CT-1
3,11,31.336877,-82.358473,8,GA,Georgia,1,GA-1,GA-1
14,21,33.473700,-82.013100,1,GA,Georgia,2,GA-2,GA-2


## BASELINE

In [17]:
# ============================================================
# BASELINE V2: GREEDY FEASIBLE CONSTRUCTOR (CURRENT PIPELINE)
# - Uses the CURRENT acd2 / claims_model / capacity_ad / planning_days
# - Keeps same structural business logic as current final model:
#     * one claim per adjuster-day
#     * onsite claims require one-cluster-per-adjuster consistency
#     * online claims do NOT require deployment
#     * team cap per claim-day
# - No optimization, just greedy assignment
# - Produces outputs in the same style as the optimization flow
# ============================================================

import pandas as pd
import numpy as np
import time
from collections import defaultdict

# ------------------------------------------------------------
# 0) SAFETY CHECKS
# ------------------------------------------------------------
required_objs = [
    "acd2", "claims_model", "planning_days",
    "claim_cluster_map", "claims_calendar",
    "adjusters_core", "cluster_summary"
]
missing_objs = [x for x in required_objs if x not in globals()]
if missing_objs:
    raise NameError(f"Missing required objects before baseline run: {missing_objs}")

# rebuild capacity_ad if missing
if "capacity_ad" not in globals():
    capacity_ad = (
        adjusters_core[["adjuster_id"]]
        .copy()
        .assign(key=1)
        .merge(planning_days[["day"]].copy().assign(key=1), on="key")
        .drop(columns="key")
    )

    capacity_ad["adjuster_id"] = capacity_ad["adjuster_id"].astype(str)
    capacity_ad["day"] = capacity_ad["day"].astype(int)
    capacity_ad["capacity_days"] = 1.0

    if "shift_capacity_days" in adjusters_core.columns:
        cap_lookup = adjusters_core[["adjuster_id", "shift_capacity_days"]].copy()
        cap_lookup["adjuster_id"] = cap_lookup["adjuster_id"].astype(str)

        capacity_ad = capacity_ad.merge(
            cap_lookup,
            on="adjuster_id",
            how="left"
        )

        capacity_ad["capacity_days"] = capacity_ad["shift_capacity_days"].fillna(1.0).astype(float)
    else:
        capacity_ad["shift_capacity_days"] = 1.0

    print("capacity_ad was missing and has been rebuilt.")
# ------------------------------------------------------------
# 1) STANDARDIZE CORE INPUTS
# ------------------------------------------------------------
claims_model_ids = claims_model.copy()

required_cols = ["claim_id", "severity", "arrival_day", "due_day", "onsite", "online", "cluster_id"]
missing = [c for c in required_cols if c not in claims_model_ids.columns]
if missing:
    raise KeyError(f"claims_model is missing required columns: {missing}")

claims_model_ids["claim_id"] = claims_model_ids["claim_id"].astype(str)
claims_model_ids["cluster_id"] = claims_model_ids["cluster_id"].astype(str)
claims_model_ids["online"] = claims_model_ids["online"].astype(bool)
claims_model_ids["onsite"] = claims_model_ids["onsite"].astype(bool)
claims_model_ids["arrival_day"] = claims_model_ids["arrival_day"].astype(int)
claims_model_ids["due_day"] = claims_model_ids["due_day"].astype(int)

if "w" not in claims_model_ids.columns:
    claims_model_ids["w"] = (
        pd.to_numeric(claims_model_ids["severity"], errors="coerce")
        .fillna(1)
        .astype(int)
    )

claim_cluster_id = dict(zip(claims_model_ids["claim_id"], claims_model_ids["cluster_id"]))
claim_is_onsite = dict(zip(claims_model_ids["claim_id"], claims_model_ids["onsite"]))
claim_is_online = dict(zip(claims_model_ids["claim_id"], claims_model_ids["online"]))
claim_due_day = dict(zip(claims_model_ids["claim_id"], claims_model_ids["due_day"]))
claim_arr_day = dict(zip(claims_model_ids["claim_id"], claims_model_ids["arrival_day"]))
claim_weight = dict(zip(claims_model_ids["claim_id"], claims_model_ids["w"]))

I = sorted(claims_model_ids["claim_id"].unique().tolist())
D = sorted(planning_days["day"].astype(int).unique().tolist())

# ------------------------------------------------------------
# 2) BUILD BASELINE CANDIDATE TABLE FROM CURRENT acd2
# ------------------------------------------------------------
work_df = acd2.copy()

needed_cols = ["adjuster_id", "claim_id", "day", "cluster_id", "progress_per_day"]
missing = [c for c in needed_cols if c not in work_df.columns]
if missing:
    raise KeyError(f"acd2 is missing required columns: {missing}")

work_df["adjuster_id"] = work_df["adjuster_id"].astype(str)
work_df["claim_id"] = work_df["claim_id"].astype(str)
work_df["cluster_id"] = work_df["cluster_id"].astype(str)
work_df["day"] = work_df["day"].astype(int)
work_df["progress_per_day"] = pd.to_numeric(work_df["progress_per_day"], errors="coerce").fillna(0.0)

# ensure optional reporting columns exist
optional_cols = [
    "claim_type", "severity", "skill_for_claim", "service_days",
    "drive_seconds", "drive_miles", "travel_hours", "travel_mult",
    "base_progress_per_day", "onsite", "online", "arrival_day", "due_day"
]
for c in optional_cols:
    if c not in work_df.columns:
        work_df[c] = np.nan

# use only feasible positive-productivity rows
work_df = work_df.loc[work_df["progress_per_day"] > 0].copy()

# merge claims metadata if missing / for consistency
work_df = work_df.drop(columns=["onsite", "online"], errors="ignore").merge(
    claims_model_ids[["claim_id", "onsite", "online", "arrival_day", "due_day", "severity", "cluster_id"]],
    on="claim_id",
    how="left",
    suffixes=("", "_claim"),
    validate="many_to_one"
)

if "cluster_id_claim" in work_df.columns:
    work_df["cluster_id"] = work_df["cluster_id"].fillna(work_df["cluster_id_claim"])
    work_df = work_df.drop(columns=["cluster_id_claim"])

# capacity
capacity_ad["adjuster_id"] = capacity_ad["adjuster_id"].astype(str)
capacity_ad["day"] = capacity_ad["day"].astype(int)

cap = {
    (a, d): float(c)
    for a, d, c in zip(
        capacity_ad["adjuster_id"],
        capacity_ad["day"],
        capacity_ad["capacity_days"]
    )
}

# candidate keys
W_keys = list(zip(work_df["adjuster_id"], work_df["claim_id"], work_df["day"]))

prog = {
    (a, i, d): float(p)
    for a, i, d, p in zip(
        work_df["adjuster_id"],
        work_df["claim_id"],
        work_df["day"],
        work_df["progress_per_day"]
    )
}

work_metrics_lookup = {
    (a, i, d): {
        "cluster_id": c,
        "claim_type": ct,
        "severity": int(sev) if pd.notna(sev) else np.nan,
        "skill_for_claim": sfc,
        "service_days": float(sd) if pd.notna(sd) else np.nan,
        "drive_seconds": float(ds) if pd.notna(ds) else 0.0,
        "drive_miles": float(dm) if pd.notna(dm) else 0.0,
        "travel_hours": float(th) if pd.notna(th) else 0.0,
        "travel_mult": float(tm) if pd.notna(tm) else np.nan,
        "base_progress_per_day": float(bp) if pd.notna(bp) else np.nan,
        "effective_progress_per_day": float(pp),
        "onsite": bool(os),
        "online": bool(ol),
        "arrival_day": int(ad) if pd.notna(ad) else np.nan,
        "due_day": int(dd) if pd.notna(dd) else np.nan,
    }
    for a, i, d, c, ct, sev, sfc, sd, ds, dm, th, tm, bp, pp, os, ol, ad, dd in zip(
        work_df["adjuster_id"],
        work_df["claim_id"],
        work_df["day"],
        work_df["cluster_id"],
        work_df["claim_type"],
        work_df["severity"],
        work_df["skill_for_claim"],
        work_df["service_days"],
        work_df["drive_seconds"],
        work_df["drive_miles"],
        work_df["travel_hours"],
        work_df["travel_mult"],
        work_df["base_progress_per_day"],
        work_df["progress_per_day"],
        work_df["onsite"],
        work_df["online"],
        work_df["arrival_day"],
        work_df["due_day"],
    )
}

# group candidate rows by claim
W_by_i = defaultdict(list)
for (a, i, d) in W_keys:
    W_by_i[i].append((a, i, d))

# ------------------------------------------------------------
# 3) GREEDY BASELINE CONSTRUCTION
# ------------------------------------------------------------
t0 = time.perf_counter()

# priority:
#   1) higher weight/severity
#   2) earlier due day
#   3) earlier arrival day
claim_order = sorted(
    I,
    key=lambda i: (-claim_weight[i], claim_due_day[i], claim_arr_day[i], i)
)

claim_progress = defaultdict(float)
claim_completion_day = {}
assigned_ad = set()                 # (adjuster_id, day)
deployed_cluster = {}               # adjuster_id -> single onsite cluster across horizon
team_count_id = defaultdict(int)    # (claim_id, day) -> # adjusters on claim-day
work_assignments = []

for i in claim_order:
    if claim_progress[i] >= 1.0:
        continue

    c = claim_cluster_id[i]
    onsite = claim_is_onsite[i]

    def candidate_sort_key(k):
        a, _, d = k

        # onsite logic prefers:
        #   already deployed to same cluster
        #   then not-yet-deployed
        #   then anything else becomes infeasible
        same_cluster_bonus = 0
        if onsite:
            current_cluster = deployed_cluster.get(a)
            if current_cluster == c:
                same_cluster_bonus = 0
            elif current_cluster is None:
                same_cluster_bonus = 1
            else:
                same_cluster_bonus = 9

        # greedy preference:
        # earlier day, then onsite feasibility, then higher progress
        return (d, same_cluster_bonus, -prog[k], a)

    cand_keys = sorted(W_by_i.get(i, []), key=candidate_sort_key)

    for (a, _, d) in cand_keys:
        if claim_progress[i] >= 1.0:
            claim_completion_day[i] = d
            break

        # one claim per adjuster-day
        if (a, d) in assigned_ad:
            continue

        # team cap per claim-day
        if team_count_id[(i, d)] >= MAX_TEAM:
            continue

        # capacity
        if cap.get((a, d), 0.0) < 1.0:
            continue

        # onsite claims require one-cluster-per-adjuster consistency
        if onsite:
            current_cluster = deployed_cluster.get(a)
            if current_cluster is not None and current_cluster != c:
                continue

        # assign
        work_assignments.append((a, i, d))
        assigned_ad.add((a, d))
        team_count_id[(i, d)] += 1
        claim_progress[i] += prog[(a, i, d)]

        if onsite:
            deployed_cluster[a] = c

        if claim_progress[i] >= 1.0:
            claim_completion_day[i] = d
            break

runtime_seconds = time.perf_counter() - t0

# ------------------------------------------------------------
# 4) MATERIALIZE BASELINE OUTPUT TABLES
# ------------------------------------------------------------
work_assignments_df = pd.DataFrame(
    work_assignments,
    columns=["adjuster_id", "claim_id", "day"]
)

if not work_assignments_df.empty:
    work_metrics_df = pd.DataFrame([
        {
            "adjuster_id": a,
            "claim_id": i,
            "day": d,
            **work_metrics_lookup[(a, i, d)]
        }
        for (a, i, d) in work_assignments
    ])
else:
    work_metrics_df = pd.DataFrame(columns=[
        "adjuster_id", "claim_id", "day", "cluster_id",
        "claim_type", "severity", "skill_for_claim", "service_days",
        "drive_seconds", "drive_miles", "travel_hours", "travel_mult",
        "base_progress_per_day", "effective_progress_per_day",
        "onsite", "online", "arrival_day", "due_day"
    ])

deployed_pairs = sorted((a, c) for a, c in deployed_cluster.items())
deployed_df = pd.DataFrame(deployed_pairs, columns=["adjuster_id", "cluster_id"])

completed_df_baseline = claims_model_ids[[
    "claim_id", "severity", "arrival_day", "due_day", "onsite", "online", "cluster_id", "w"
]].copy()

completed_df_baseline["completed"] = completed_df_baseline["claim_id"].map(
    lambda x: claim_progress[x] >= 1.0
)
completed_df_baseline["completion_day"] = completed_df_baseline["claim_id"].map(claim_completion_day)
completed_df_baseline["days_to_complete"] = np.where(
    completed_df_baseline["completed"],
    completed_df_baseline["completion_day"] - completed_df_baseline["arrival_day"] + 1,
    np.nan
)
completed_df_baseline["cycle_time_days_baseline"] = np.where(
    completed_df_baseline["completed"],
    completed_df_baseline["completion_day"] - completed_df_baseline["arrival_day"],
    np.nan
)
completed_df_baseline["completed_on_time"] = completed_df_baseline["completed"]

# claim summary
if not work_metrics_df.empty:
    claim_work_summary = (
        work_metrics_df.groupby("claim_id", as_index=False)
        .agg(
            first_work_day=("day", "min"),
            last_work_day=("day", "max"),
            total_workdays=("day", "count"),
            n_adjusters_used=("adjuster_id", "nunique"),
            total_effective_progress=("effective_progress_per_day", "sum"),
            total_base_progress=("base_progress_per_day", "sum"),
            total_productivity_loss=(
                "effective_progress_per_day",
                lambda s: np.nan
            ),
            total_travel_hours=("travel_hours", "sum"),
            total_drive_miles=("drive_miles", "sum")
        )
    )

    # rebuild productivity loss from work metrics
    loss_df = (
        work_metrics_df.assign(
            productivity_loss_abs=lambda d: d["base_progress_per_day"] - d["effective_progress_per_day"]
        )
        .groupby("claim_id", as_index=False)["productivity_loss_abs"]
        .sum()
        .rename(columns={"productivity_loss_abs": "total_productivity_loss"})
    )

    claim_work_summary = claim_work_summary.drop(columns=["total_productivity_loss"]).merge(
        loss_df,
        on="claim_id",
        how="left"
    )
else:
    claim_work_summary = pd.DataFrame(columns=[
        "claim_id", "first_work_day", "last_work_day", "total_workdays",
        "n_adjusters_used", "total_effective_progress", "total_base_progress",
        "total_productivity_loss", "total_travel_hours", "total_drive_miles"
    ])

completed_df_baseline = completed_df_baseline.merge(
    claim_work_summary,
    on="claim_id",
    how="left"
)

# assignments-style table
assignments_df_baseline = work_metrics_df.copy()

assignments_df_baseline = assignments_df_baseline.merge(
    adjusters_core[[
        "adjuster_id", "skill_business", "skill_personal",
        "will_travel", "home_city", "home_state", "home_lat", "home_lon"
    ]].assign(adjuster_id=lambda d: d["adjuster_id"].astype(str)),
    on="adjuster_id",
    how="left"
)

assignments_df_baseline["modality"] = np.where(assignments_df_baseline["onsite"], "onsite", "online")
assignments_df_baseline["productivity_loss_pct"] = 1 - assignments_df_baseline["travel_mult"]
assignments_df_baseline["productivity_loss_abs"] = (
    assignments_df_baseline["base_progress_per_day"] - assignments_df_baseline["effective_progress_per_day"]
)

# deployments-style table
deployments_df_baseline = deployed_df.copy()
if not deployments_df_baseline.empty:
    deployments_df_baseline["adjuster_id"] = deployments_df_baseline["adjuster_id"].astype(str)
    deployments_df_baseline["cluster_id"] = deployments_df_baseline["cluster_id"].astype(str)

    deployments_df_baseline = deployments_df_baseline.merge(
        adjusters_core[[
            "adjuster_id", "skill_business", "skill_personal",
            "will_travel", "home_city", "home_state", "home_lat", "home_lon"
        ]].assign(adjuster_id=lambda d: d["adjuster_id"].astype(str)),
        on="adjuster_id",
        how="left"
    )

    deployments_df_baseline = deployments_df_baseline.merge(
        cluster_summary[[
            "cluster_id", "claims", "onsite_claims", "online_claims", "has_onsite"
        ]].assign(cluster_id=lambda d: d["cluster_id"].astype(str)),
        on="cluster_id",
        how="left"
    )
else:
    deployments_df_baseline = pd.DataFrame(columns=[
        "adjuster_id", "cluster_id", "skill_business", "skill_personal",
        "will_travel", "home_city", "home_state", "home_lat", "home_lon",
        "claims", "onsite_claims", "online_claims", "has_onsite"
    ])

# adjuster-day baseline
adjusters_df_baseline = adjusters_core.copy()
adjusters_df_baseline["adjuster_id"] = adjusters_df_baseline["adjuster_id"].astype(str)

adjuster_day_df_baseline = (
    adjusters_df_baseline[["adjuster_id"]]
    .assign(key=1)
    .merge(planning_days.assign(key=1), on="key")
    .drop(columns="key")
)

adjuster_day_df_baseline["adjuster_id"] = adjuster_day_df_baseline["adjuster_id"].astype(str)
adjuster_day_df_baseline["day"] = adjuster_day_df_baseline["day"].astype(int)
adjuster_day_df_baseline["capacity_days"] = 1.0

if not assignments_df_baseline.empty:
    adjuster_day_df_baseline = adjuster_day_df_baseline.merge(
        assignments_df_baseline[[
            "adjuster_id", "day", "claim_id",
            "modality", "effective_progress_per_day",
            "productivity_loss_abs"
        ]],
        on=["adjuster_id", "day"],
        how="left"
    )
else:
    for c in ["claim_id", "modality", "effective_progress_per_day", "productivity_loss_abs"]:
        adjuster_day_df_baseline[c] = np.nan

adjuster_day_df_baseline["assigned_flag"] = adjuster_day_df_baseline["claim_id"].notna().astype(int)

# ------------------------------------------------------------
# 5) METRICS
# ------------------------------------------------------------
n_claims_total = len(completed_df_baseline)
n_claims_completed = int(completed_df_baseline["completed"].sum())

pct_completed_within_sla = (
    100.0 * completed_df_baseline["completed_on_time"].mean()
    if n_claims_total > 0 else np.nan
)

weighted_completed = completed_df_baseline.loc[completed_df_baseline["completed"], "w"].sum()
weighted_total = completed_df_baseline["w"].sum()
weighted_completion_pct = (
    100.0 * weighted_completed / weighted_total
    if weighted_total > 0 else np.nan
)

avg_cycle_time_days = completed_df_baseline["cycle_time_days_baseline"].mean()
median_cycle_time_days = completed_df_baseline["cycle_time_days_baseline"].median()

used_adjuster_days = len(work_assignments_df)
total_capacity_days = float(capacity_ad["capacity_days"].sum())
adjuster_utilization_pct = (
    100.0 * used_adjuster_days / total_capacity_days
    if total_capacity_days > 0 else np.nan
)

all_adjusters = sorted(capacity_ad["adjuster_id"].astype(str).unique().tolist())
adjuster_days_worked = (
    work_assignments_df.groupby("adjuster_id").size()
    .reindex(all_adjusters, fill_value=0)
    .reset_index(name="days_worked")
    .rename(columns={"index": "adjuster_id"})
)

workload_std_days = adjuster_days_worked["days_worked"].std(ddof=0)
workload_mean_days = adjuster_days_worked["days_worked"].mean()
workload_cv_days = (
    workload_std_days / workload_mean_days
    if workload_mean_days > 0 else np.nan
)

total_travel_hours = assignments_df_baseline["travel_hours"].sum() if not assignments_df_baseline.empty else 0.0
total_drive_miles = assignments_df_baseline["drive_miles"].sum() if not assignments_df_baseline.empty else 0.0
deployment_count = len(deployments_df_baseline)
deployed_adjuster_count = deployments_df_baseline["adjuster_id"].nunique() if not deployments_df_baseline.empty else 0

total_productivity_loss = (
    assignments_df_baseline["productivity_loss_abs"].sum()
    if not assignments_df_baseline.empty else 0.0
)
avg_productivity_loss_pct = (
    100.0 * assignments_df_baseline["productivity_loss_pct"].mean()
    if not assignments_df_baseline.empty else 0.0
)

allocation_runtime_seconds = runtime_seconds

baseline_metrics = {
    "n_claims_total": n_claims_total,
    "n_claims_completed_within_sla": n_claims_completed,
    "pct_completed_within_sla": pct_completed_within_sla,
    "weighted_completion_pct": weighted_completion_pct,
    "avg_cycle_time_days": avg_cycle_time_days,
    "median_cycle_time_days": median_cycle_time_days,
    "adjuster_utilization_pct": adjuster_utilization_pct,
    "workload_std_days": workload_std_days,
    "workload_cv_days": workload_cv_days,
    "total_travel_hours": total_travel_hours,
    "total_drive_miles": total_drive_miles,
    "deployment_count": deployment_count,
    "deployed_adjuster_count": deployed_adjuster_count,
    "total_productivity_loss": total_productivity_loss,
    "avg_productivity_loss_pct": avg_productivity_loss_pct,
    "allocation_runtime_seconds": allocation_runtime_seconds,
    "used_adjuster_days": used_adjuster_days,
    "total_capacity_days": total_capacity_days,
}

baseline_metrics_df = pd.DataFrame({
    "metric": list(baseline_metrics.keys()),
    "value": list(baseline_metrics.values())
})

# adjuster-claim baseline summary
if not assignments_df_baseline.empty:
    adjuster_claim_list_baseline = (
        assignments_df_baseline.groupby(["adjuster_id", "claim_id"], as_index=False)
        .agg(
            days_worked=("day", "count"),
            total_effective_progress=("effective_progress_per_day", "sum"),
            total_travel_hours=("travel_hours", "sum"),
            total_drive_miles=("drive_miles", "sum"),
            total_productivity_loss=("productivity_loss_abs", "sum"),
        )
        .merge(
            completed_df_baseline[["claim_id", "severity", "onsite", "online", "cluster_id", "due_day"]],
            on="claim_id",
            how="left"
        )
        .sort_values(["days_worked", "severity"], ascending=[False, False])
        .reset_index(drop=True)
    )
else:
    adjuster_claim_list_baseline = pd.DataFrame(columns=[
        "adjuster_id", "claim_id", "days_worked", "total_effective_progress",
        "total_travel_hours", "total_drive_miles", "total_productivity_loss",
        "severity", "onsite", "online", "cluster_id", "due_day"
    ])

# ------------------------------------------------------------
# 6) QUICK PRINTS
# ------------------------------------------------------------
print("==================================================")
print("BASELINE V2 RESULTS")
print("==================================================")
print(f"Claims completed within SLA: {n_claims_completed} / {n_claims_total} ({pct_completed_within_sla:.2f}%)")
print(f"Weighted completion %     : {weighted_completion_pct:.2f}%")
print(f"Average cycle time (days) : {avg_cycle_time_days:.2f}")
print(f"Median cycle time (days)  : {median_cycle_time_days:.2f}")
print(f"Adjuster utilization %    : {adjuster_utilization_pct:.2f}%")
print(f"Workload CV (days worked) : {workload_cv_days:.4f}")
print(f"Total travel hours        : {total_travel_hours:.2f}")
print(f"Total drive miles         : {total_drive_miles:.2f}")
print(f"Deployment count          : {deployment_count}")
print(f"Runtime (seconds)         : {allocation_runtime_seconds:.4f}")

print("\nTop 20 baseline adjuster-claim assignments:")
print(adjuster_claim_list_baseline.head(20))

BASELINE V2 RESULTS
Claims completed within SLA: 938 / 1053 (89.08%)
Weighted completion %     : 84.46%
Average cycle time (days) : 1.22
Median cycle time (days)  : 0.00
Adjuster utilization %    : 9.24%
Workload CV (days worked) : 0.8459
Total travel hours        : 4369.62
Total drive miles         : 168517.91
Deployment count          : 552
Runtime (seconds)         : 9.1996

Top 20 baseline adjuster-claim assignments:
   adjuster_id  claim_id  days_worked  total_effective_progress  \
0     28980270  63933354            5                     0.375   
1     29036599  63933354            5                     0.375   
2     54675071  21868312            5                     0.375   
3     55121511  21868312            5                     0.375   
4     77703494  46524537            5                     0.250   
5     78072257  46524537            5                     0.250   
6     52192658  75087575            4                     0.300   
7     85342366  11362451            4  